# 📐 03 · Vectors and matrices: the shape of biological data
### Dot products, correlation, matrix products, solving systems, rank, eigenvectors and projections — the linear algebra behind regression, PCA and networks

**BiologyToML** · Part A — Foundations: from biology to data · Module 03 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/03_linear_algebra_for_data.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy (`np.linalg`), pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | Modules 01–02 (NumPy arrays, `axis=`, mean, variance, covariance and correlation). No linear-algebra course needed. |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | prerequisite (02-680 *Essential Mathematics & Statistics*); `002_regression` slides 26–40 & 46; `011_PCA_mz_031825` slides 18–20 & 37–38 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Real data: the Golub et al. (1999) leukaemia expression matrix (72 patients × 3,571 genes) and the Stamey et al. (1989)
> prostate-cancer table (97 men × 8 clinical measurements) used on the 02-620 regression slides. The time-course genes,
> the polygenic-score toy, the cell-type deconvolution example and the genotype matrices of Sections 5 and 8 are
> SIMULATED and labelled as such.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Represent** a gene's expression across patients and a patient's profile across genes as vectors, and **compute** sums, scalings, lengths (norms) and dot products by hand and in NumPy.
2. **Explain** the geometry of the dot product (angle, projection) and **show** that Pearson correlation is the cosine of the angle between *centred* vectors; **choose** between Euclidean and correlation distance.
3. **Compute and interpret** matrix–vector and matrix–matrix products: a polygenic score $X\mathbf w$, the augmented intercept column, the cross-product matrix $X^\top X$ and the covariance $S = X^\top X/N$.
4. **Solve** linear systems with `np.linalg.solve` (cell-type deconvolution, the normal equations) and **explain** why we avoid forming an explicit inverse.
5. **Diagnose** singular matrices through rank — perfect linkage disequilibrium, the allele-coding trap and $N<J$ — the exact failure of least squares on `002_regression` slide 46.
6. **Compute** eigenvalues and eigenvectors of a symmetric matrix with `np.linalg.eigh`, **verify** $S=UDU^\top$ and $U^\top U=I$, and **draw** the axes of a covariance ellipse (`011_PCA` slides 18–20).
7. **Change coordinates** by projecting onto orthonormal axes (`011_PCA` slides 37–38) and **show** that adding $\lambda I$ makes $X^\top X$ invertible (the seed of ridge regression).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, load Golub and prostate data | 5 min |
| 1 · Vectors | addition, scaling, norm, dot product, angle; correlation = cosine of centred vectors | 20 min |
| 2 · Distances | Euclidean vs correlation distance; which patients resemble each other? | 15 min |
| 3 · Matrices as data tables | $X\mathbf w$ as a polygenic score, intercept column, products, $X^\top X$ and covariance | 25 min |
| 4 · Solving linear systems | inverse, `solve`, cell-type deconvolution, normal equations on prostate data | 15 min |
| 5 · Rank and singularity | perfect LD, allele-coding trap, $N<J$ and infinitely many perfect fits | 20 min |
| 6 · Eigenvalues and eigenvectors | directions that are only stretched; covariance ellipse of two genes; spectrum | 25 min |
| 7 · Projections and new coordinates | walking along new axes; PCA in miniature; least squares as projection | 15 min |
| 8 · Adding $\lambda I$ | eigenvalues shift by $\lambda$: invertible even when $N<J$ | 5 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

Every data set in 02-620 is a **table of numbers**. Golub and colleagues measured about 7,000 genes in 72 leukaemia
patients (3,571 genes after standard filtering); a genome-wide association study (GWAS) records the genotypes of
hundreds of thousands of SNPs in thousands of people; Novembre *et al.* (2008) summarised 197,146 SNPs in 1,387 Europeans.
Biologists ask questions about such tables all the time, and each question is secretly a piece of **linear algebra**:

| Biological question | Linear-algebra operation | Where in 02-620 |
|---|---|---|
| Do these two genes rise and fall together across patients? | angle between two vectors (correlation) | clustering, GGMs |
| Which patients have similar expression profiles? | distances between rows | k-NN, clustering |
| What is this person's genetic risk score? | a matrix–vector product $X\mathbf w$ | linear/logistic regression |
| Which combination of SNPs best predicts blood pressure? | solving $X^\top X\,\mathbf w = X^\top\mathbf y$ | regression (`002` slides 35–38) |
| Why does that fail with 3 million SNPs and 1,000 people? | rank of $X^\top X$ | `002` slide 46 → ridge, lasso |
| What are the main axes of variation among people? | eigenvectors of the covariance matrix | PCA (`011` slides 17–54) |

The 02-620 slides assume you can read $\mathbf y = X\mathbf w$, $(X^\top X)^{-1}$ or $S=UDU^\top$ fluently. This notebook
builds that fluency from zero, on real expression and clinical data, so that the regression, PCA and graphical-model
lectures read like plain sentences.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They load the Golub leukaemia and prostate-cancer data (with an offline fallback), simulate Hardy–Weinberg genotypes, and z-score columns. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import io
import os
import urllib.request
import numpy as np
import pandas as pd

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


PROSTATE_CSV = """lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa,train
-0.579818,2.76946,50,-1.38629,0,-1.38629,6,0,-0.430783,T
-0.994252,3.31963,58,-1.38629,0,-1.38629,6,0,-0.162519,T
-0.510826,2.69124,74,-1.38629,0,-1.38629,7,20,-0.162519,T
-1.20397,3.28279,58,-1.38629,0,-1.38629,6,0,-0.162519,T
0.751416,3.43237,62,-1.38629,0,-1.38629,6,0,0.371564,T
-1.04982,3.22883,50,-1.38629,0,-1.38629,6,0,0.765468,T
0.737164,3.47352,64,0.615186,0,-1.38629,6,0,0.765468,F
0.693147,3.53951,58,1.53687,0,-1.38629,6,0,0.854415,T
-0.776529,3.53951,47,-1.38629,0,-1.38629,6,0,1.04732,F
0.223144,3.24454,63,-1.38629,0,-1.38629,6,0,1.04732,F
0.254642,3.60414,65,-1.38629,0,-1.38629,6,0,1.26695,T
-1.34707,3.59868,63,1.26695,0,-1.38629,6,0,1.26695,T
1.61343,3.02286,63,-1.38629,0,-0.597837,7,30,1.26695,T
1.47705,2.99823,67,-1.38629,0,-1.38629,7,5,1.34807,T
1.20597,3.44202,57,-1.38629,0,-0.430783,7,5,1.39872,F
1.54116,3.06105,66,-1.38629,0,-1.38629,6,0,1.44692,T
-0.415515,3.51601,70,1.24415,0,-0.597837,7,30,1.47018,T
2.28849,3.64936,66,-1.38629,0,0.371564,6,0,1.4929,T
-0.562119,3.26767,41,-1.38629,0,-1.38629,6,0,1.55814,T
0.182322,3.82538,70,1.65823,0,-1.38629,6,0,1.59939,T
1.1474,3.41936,59,-1.38629,0,-1.38629,6,0,1.639,T
2.05924,3.50104,60,1.47476,0,1.34807,7,20,1.65823,F
-0.544727,3.37588,59,-0.798508,0,-1.38629,6,0,1.69562,T
1.78171,3.45157,63,0.438255,0,1.17866,7,60,1.7138,T
0.385262,3.6674,69,1.59939,0,-1.38629,6,0,1.73166,F
1.44692,3.12457,68,0.300105,0,-1.38629,6,0,1.76644,F
0.512824,3.71965,65,-1.38629,0,-0.798508,7,70,1.80006,T
-0.400478,3.86598,67,1.81645,0,-1.38629,7,20,1.81645,F
1.04028,3.12895,67,0.223144,0,0.0487902,7,80,1.84845,T
2.40964,3.37588,65,-1.38629,0,1.61939,6,0,1.89462,T
0.285179,4.09017,65,1.96291,0,-0.798508,6,0,1.92425,T
0.182322,3.80444,65,1.70475,0,-1.38629,6,0,2.00821,F
1.27536,3.03735,71,1.26695,0,-1.38629,6,0,2.00821,T
0.00995033,3.26767,54,-1.38629,0,-1.38629,6,0,2.02155,F
-0.0100503,3.21687,63,-1.38629,0,-0.798508,6,0,2.04769,T
1.30833,4.11985,64,2.17134,0,-1.38629,7,5,2.08567,F
1.42311,3.65713,73,-0.579819,0,1.65823,8,15,2.15756,T
0.457425,2.37491,64,-1.38629,0,-1.38629,7,15,2.19165,T
2.66096,4.08514,68,1.37372,1,1.83258,7,35,2.21375,T
0.797507,3.01308,56,0.936093,0,-0.162519,7,5,2.27727,T
0.620576,3.142,60,-1.38629,0,-1.38629,9,80,2.29757,T
1.4422,3.68261,68,-1.38629,0,-1.38629,7,10,2.30757,F
0.582216,3.86598,62,1.7138,0,-0.430783,6,0,2.32728,T
1.77156,3.89691,61,-1.38629,0,0.81093,7,6,2.37491,F
1.48614,3.4095,66,1.7492,0,-0.430783,7,20,2.52172,T
1.66393,3.39283,61,0.615186,0,-1.38629,7,15,2.55334,T
2.72785,3.99545,79,1.87947,1,2.65676,9,100,2.56879,T
1.16315,4.03512,68,1.7138,0,-0.430783,7,40,2.56879,F
1.74572,3.49802,43,-1.38629,0,-1.38629,6,0,2.59152,F
1.22083,3.56812,70,1.37372,0,-0.798508,6,0,2.59152,F
1.09192,3.9936,68,-1.38629,0,-1.38629,7,50,2.65676,T
1.66013,4.23483,64,2.07317,0,-1.38629,6,0,2.67759,T
0.512824,3.63363,64,1.4929,0,0.0487902,7,70,2.68444,F
2.12704,4.12147,68,1.76644,0,1.44692,7,40,2.69124,F
3.15359,3.51601,59,-1.38629,0,-1.38629,7,5,2.70471,F
1.26695,4.28013,66,2.12226,0,-1.38629,7,15,2.718,T
0.97456,2.86505,47,-1.38629,0,0.500775,7,4,2.78809,F
0.463734,3.76468,49,1.42311,0,-1.38629,6,0,2.79423,T
0.542324,4.17823,70,0.438255,0,-1.38629,7,20,2.80639,T
1.06126,3.85121,61,1.29473,0,-1.38629,7,40,2.81241,T
0.457425,4.5245,73,2.3263,0,-1.38629,6,0,2.842,T
1.99742,3.71965,63,1.61939,1,1.90954,7,40,2.85359,F
2.77571,3.52489,72,-1.38629,0,1.55814,9,95,2.85359,T
2.03471,3.91701,66,2.00821,1,2.11021,7,60,2.882,F
2.07317,3.62301,64,-1.38629,0,-1.38629,6,0,2.882,F
1.45862,3.83622,61,1.32176,0,-0.430783,7,20,2.88759,F
2.02287,3.87847,68,1.78339,0,1.32176,7,70,2.92047,T
2.19834,4.05091,72,2.30757,0,-0.430783,7,10,2.96269,T
-0.446287,4.40855,69,-1.38629,0,-1.38629,6,0,2.96269,T
1.19392,4.78038,72,2.3263,0,-0.798508,7,5,2.97298,T
1.86408,3.59319,60,-1.38629,1,1.32176,7,60,3.01308,T
1.16002,3.34109,77,1.7492,0,-1.38629,7,25,3.03735,T
1.21491,3.82538,69,-1.38629,1,0.223144,7,20,3.05636,F
1.83896,3.23672,60,0.438255,1,1.17866,9,90,3.07501,F
2.99923,3.84908,69,-1.38629,1,1.90954,7,20,3.27526,T
3.14113,3.26385,68,-0.0512933,1,2.42037,7,50,3.33755,T
2.01089,4.43379,72,2.12226,0,0.500775,7,60,3.39283,T
2.53766,4.35478,78,2.3263,0,-1.38629,7,10,3.4356,T
2.6483,3.58213,69,-1.38629,1,2.584,7,70,3.45789,T
2.77944,3.82319,63,-1.38629,0,0.371564,7,50,3.51304,F
1.46787,3.07038,66,0.559616,0,0.223144,7,40,3.51601,T
2.51366,3.47352,57,0.438255,0,2.32728,7,60,3.53076,T
2.61301,3.88875,77,-0.527633,1,0.559616,7,30,3.5653,T
2.67759,3.83838,65,1.11514,0,1.7492,9,70,3.57094,F
1.56235,3.70991,60,1.69562,0,0.81093,7,30,3.58768,T
3.30285,3.51898,64,-1.38629,1,2.32728,7,60,3.63099,T
2.02419,3.7317,58,1.639,0,-1.38629,6,0,3.68009,T
1.73166,3.36902,62,-1.38629,1,0.300105,7,30,3.71235,T
2.80759,4.71805,65,-1.38629,1,2.46385,7,60,3.98434,T
1.56235,3.69511,76,0.936093,1,0.81093,7,75,3.9936,T
3.24649,4.10182,68,-1.38629,0,-1.38629,6,0,4.02981,T
2.5329,3.67757,61,1.34807,1,-1.38629,7,15,4.12955,T
2.83027,3.8764,68,-1.38629,1,1.32176,7,60,4.38515,T
3.821,3.89691,44,-1.38629,1,2.16905,7,40,4.68444,T
2.90745,3.39619,52,-1.38629,1,2.46385,7,10,5.14312,F
2.88256,3.77391,68,1.55814,1,1.55814,7,80,5.47751,T
3.47197,3.975,68,0.438255,1,2.90417,7,20,5.58293,F
"""


def load_prostate() -> pd.DataFrame:
    """Prostate cancer data of 02-620 Lecture 2 (97 men, 8 clinical predictors, outcome ``lpsa``)."""
    df = pd.read_csv(io.StringIO(PROSTATE_CSV))
    df["train"] = df["train"] == "T"
    return df


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'load_prostate', 'simulate_golub_like', 'load_golub', 'simulate_genotypes', 'standardize']))

In [ ]:
# ── 0.1 Load the two real data sets of this module ──
golub = load_golub()
print("Golub source:", golub["source"])
print("expression matrix:", golub["X"].shape, "= N patients × J genes (log10 expression units)")
names_, counts_ = np.unique(golub["labels"], return_counts=True)
print("labels:", {str(k): int(v) for k, v in zip(names_, counts_)})

prostate = load_prostate()
FEATURES = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
print("\nprostate table:", prostate.shape, "→ 8 clinical predictors + outcome lpsa + train flag")

# Affymetrix annotations of the few probes we use by name (HuGeneFL array)
GENE_NOTES = {"M31523_at": "TCF3 (E2A, B-cell transcription factor)", "M92287_at": "CCND3 (cyclin D3)",
              "M23197_at": "CD33 (myeloid surface antigen)", "U22376_cds2_s_at": "MYB (c-myb)"}
GENE_INDEX = {g: i for i, g in enumerate(golub["genes"])}
if golub["source"].startswith("SIMULATED"):
    # Offline stand-in: the real probe IDs are absent, so map each probe named in this notebook to a
    # stand-in gene that differs between the classes. Every cell still runs; the numbers differ from the text.
    _order = np.argsort(np.abs(golub["X"][golub["y"] == 1].mean(0) - golub["X"][golub["y"] == 0].mean(0)))[::-1]
    for _k, _probe in enumerate(["M31523_at", "M92287_at", "M23197_at", "U22376_cds2_s_at"]):
        GENE_INDEX[_probe] = int(_order[_k])
for probe, note in GENE_NOTES.items():
    print(f"  {probe:18s} → column {GENE_INDEX.get(probe, 'n/a')}: {note}")

🔍 **What just happened?** We loaded a matrix with **N = 72 rows (patients)** and **J = 3,571 columns (genes)** — the
course-wide convention (samples in rows, features in columns, as on `002_regression` slide 26). Values are log10
expression; about one value in five sits exactly at 2.0 because the standard preprocessing (Dudoit *et al.* 2002) floors
raw intensities at 100 units before taking logs. We also loaded the 97-man prostate table that 02-620 uses for
multivariate regression. If you are offline, the Golub loader prints a warning and returns a SIMULATED stand-in; every
cell still runs, but the numbers quoted below refer to the real data.

---
## 1 · Vectors: lists of numbers with a geometry

> 📍 **Where we are.** Module 01 taught you NumPy arrays and Module 02 taught you means, variances and correlations.
> Here we give those arrays a **geometry** — lengths, angles, directions — because every 02-620 algorithm thinks about
> data geometrically: regression fits a direction, PCA finds directions of large spread, k-NN measures distances.

🧮 **Math refresher — vectors.** A **vector** is an ordered list of numbers. We write it as a **column**
$\mathbf x = (x_1, x_2, \dots, x_N)^\top$; the little $^\top$ (**transpose**) turns the row we can print on a page into
a column. Bold lower-case letters are vectors, $x_i$ is the $i$-th entry ("coordinate"), and $N$ is the length. The
Golub table can be read as vectors in two ways:

* a **gene vector**: one column, the expression of one gene in all $N = 72$ patients (a point in 72-dimensional space);
* a **patient vector**: one row, the expression profile of one patient across all $J = 3{,}571$ genes.

Four operations are all you need:

$$\mathbf x + \mathbf y = \begin{pmatrix} x_1 + y_1 \\ \vdots \\ x_N + y_N\end{pmatrix}, \qquad
c\,\mathbf x = \begin{pmatrix} c\,x_1 \\ \vdots \\ c\,x_N\end{pmatrix}.$$

*In words:* vectors are added entry by entry, and multiplying by a number $c$ (a **scalar**) multiplies every entry.
Converting log10 expression to log2 is exactly a scalar multiplication by $\log_2 10 \approx 3.32$.

$$\|\mathbf x\| = \sqrt{x_1^2 + x_2^2 + \dots + x_N^2} = \sqrt{\sum_{i=1}^{N} x_i^2}.$$

*In words:* the **length** (or **norm**) of a vector is Pythagoras' theorem in $N$ dimensions: square every entry, add
them up (the symbol $\sum_{i=1}^{N}$ means "add over $i = 1, \dots, N$"), take the square root.

$$\mathbf x^\top \mathbf y = x_1 y_1 + x_2 y_2 + \dots + x_N y_N = \sum_{i=1}^{N} x_i\,y_i .$$

*In words:* the **dot product** (inner product) multiplies matching entries and adds the results; it turns two vectors
into a single number. Note $\mathbf x^\top\mathbf x = \|\mathbf x\|^2$.

In [ ]:
# ── 1.1 The four operations on two real gene vectors (first 5 patients only, so you can check by hand) ──
X_g = golub["X"]
x = X_g[:5, GENE_INDEX["M31523_at"]].round(2)          # TCF3 in patients 1–5 (rounded for readability)
y = X_g[:5, GENE_INDEX["M92287_at"]].round(2)          # CCND3 in the same 5 patients
print("x (TCF3, log10)  =", x)
print("y (CCND3, log10) =", y)
print("x + y            =", x + y)
print("x in log2 units  =", (np.log2(10) * x).round(2), " (scalar multiplication by log2(10) = 3.32)")

norm_by_hand = np.sqrt(np.sum(x ** 2))
print(f"\n‖x‖ by the formula = {norm_by_hand:.4f}   np.linalg.norm = {np.linalg.norm(x):.4f}")

dot_loop = 0.0
for xi, yi in zip(x, y):                                # the definition, one term at a time
    dot_loop += xi * yi
print(f"xᵀy: loop = {dot_loop:.4f}   np.sum(x*y) = {np.sum(x * y):.4f}   x @ y = {x @ y:.4f}")

🔍 **What just happened?** Three ways of computing the dot product — an explicit loop, `np.sum(x * y)` and the `@`
operator — give the same number. In this course `@` is the matrix-multiplication operator; for two 1-D arrays it is the
dot product. Notice that the length of a 5-entry vector of values around 3 is about $\sqrt{5\times 3^2}\approx 6.7$:
lengths grow with the number of entries, which is why we will soon divide by lengths.

### 1.2 Geometry: the angle between two vectors

In two dimensions you can draw vectors as arrows. The dot product measures how much two arrows point the same way:

$$\cos\theta = \frac{\mathbf x^\top \mathbf y}{\|\mathbf x\|\,\|\mathbf y\|}, \qquad -1 \le \cos\theta \le 1 .$$

*In words:* divide the dot product by both lengths and you get the cosine of the angle $\theta$ between the arrows:
$+1$ = same direction, $0$ = perpendicular (**orthogonal**, $\mathbf x^\top\mathbf y = 0$), $-1$ = opposite directions.
The same formula works in 72 or 3,571 dimensions, where we cannot draw but can still compute.

A special case returns in PCA: if $\mathbf u$ has length 1 (a **unit vector**), then $\mathbf x^\top\mathbf u = \|\mathbf x\|\cos\theta$ is the length of the **shadow** (projection) of $\mathbf x$ on the line through $\mathbf u$
(`011_PCA` slide 32).

In [ ]:
# ── 1.2 Draw vectors, their sum and the angle between them ──
def angle_deg(a: np.ndarray, b: np.ndarray) -> float:
    """Angle between two vectors in degrees, from cos θ = aᵀb / (‖a‖‖b‖)."""
    cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))))


a, b = np.array([3.0, 1.0]), np.array([1.0, 2.0])
u = a / np.linalg.norm(a)                       # unit vector along a
shadow = (b @ u) * u                            # projection of b onto the line through a
print(f"aᵀb = {a @ b:.1f}, ‖a‖ = {np.linalg.norm(a):.3f}, ‖b‖ = {np.linalg.norm(b):.3f}, angle = {angle_deg(a, b):.1f}°")
print(f"length of b's shadow on a: bᵀu = {b @ u:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax in axes:
    ax.axhline(0, color="0.8", lw=0.8); ax.axvline(0, color="0.8", lw=0.8); ax.set_aspect("equal")
ax = axes[0]
for v, c, lab in [(a, "tab:blue", "a = (3, 1)"), (b, "tab:orange", "b = (1, 2)"), (a + b, "tab:green", "a + b")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=c, lw=2))
    ax.text(*(v * 1.05), lab, color=c, fontsize=9)
ax.plot([a[0], a[0] + b[0]], [a[1], a[1] + b[1]], ":", color="0.5"); ax.plot([b[0], a[0] + b[0]], [b[1], a[1] + b[1]], ":", color="0.5")
ax.set_xlim(-0.5, 4.8); ax.set_ylim(-0.5, 3.6); ax.set_title("adding vectors = walking one after the other")
ax = axes[1]
for v, c, lab in [(a, "tab:blue", "a"), (b, "tab:orange", "b")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=c, lw=2)); ax.text(*(v * 1.05), lab, color=c)
ax.annotate("", xy=shadow, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=2.5))
ax.plot([b[0], shadow[0]], [b[1], shadow[1]], "k--", lw=1)
ax.text(shadow[0] - 0.2, shadow[1] - 0.45, "shadow of b on a\n(length bᵀu)", color="tab:red", fontsize=8)
ax.set_xlim(-0.5, 3.6); ax.set_ylim(-0.6, 2.6); ax.set_title(f"angle θ = {angle_deg(a, b):.0f}°;  projection of b onto a")
for ax in axes:
    ax.set_xlabel("coordinate 1"); ax.set_ylabel("coordinate 2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For $\mathbf a = (3,1)$ and $\mathbf b = (1,2)$: $\mathbf a^\top\mathbf b = 3 + 2 = 5$,
$\|\mathbf a\| = \sqrt{10}$, $\|\mathbf b\| = \sqrt5$, so $\cos\theta = 5/\sqrt{50} \approx 0.707$ and $\theta = 45°$. The
red arrow is the **projection** of $\mathbf b$ onto the direction of $\mathbf a$: the dashed line from $\mathbf b$ to its
shadow meets $\mathbf a$ at a right angle. Projection onto a direction is the single most important geometric operation
in this course — PCA (Module 14) projects every patient onto a few chosen directions, and least squares (Section 7)
projects the phenotype vector onto the space spanned by the genotype columns.

### 1.3 Pearson correlation is a cosine — of *centred* vectors

Module 02 defined the correlation of two variables. Written with vectors it is

$$r(\mathbf x, \mathbf y) = \frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum_i (x_i-\bar x)^2}\,\sqrt{\sum_i (y_i-\bar y)^2}}
= \frac{\tilde{\mathbf x}^\top \tilde{\mathbf y}}{\|\tilde{\mathbf x}\|\,\|\tilde{\mathbf y}\|},
\qquad \tilde{\mathbf x} = \mathbf x - \bar x\,\mathbf 1 .$$

*In words:* subtract each gene's mean ($\bar x$, times the all-ones vector $\mathbf 1$) to get the **centred** vector
$\tilde{\mathbf x}$; the correlation is then just the cosine of the angle between the two centred vectors. Correlation
$= +1$: the centred vectors point the same way; $0$: perpendicular; $-1$: opposite.

In [ ]:
# ── 1.3 Raw cosine vs centred cosine vs Pearson r for gene pairs across all 72 patients ──
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine of the angle between two vectors."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


tcf3 = X_g[:, GENE_INDEX["M31523_at"]]
pairs = {"TCF3 vs CCND3": X_g[:, GENE_INDEX["M92287_at"]],
         "TCF3 vs CD33": X_g[:, GENE_INDEX["M23197_at"]],
         "TCF3 vs gene #100": X_g[:, 100]}
rows = []
for name, other in pairs.items():
    rows.append(dict(pair=name,
                     raw_cosine=cosine(tcf3, other),
                     centred_cosine=cosine(tcf3 - tcf3.mean(), other - other.mean()),
                     pearson_r=np.corrcoef(tcf3, other)[0, 1]))
print(pd.DataFrame(rows).round(3).to_string(index=False))

🔍 **What just happened?** The **centred cosine equals Pearson's $r$** in every row (to all printed digits) — the
formula is literally the same. The **raw** cosine, in contrast, is close to 1 for every pair, even for TCF3 vs CD33 whose
correlation is strongly negative. Why? All log-expression values are positive (between 2 and 4.2), so every raw gene
vector points into the same "corner" of 72-dimensional space, and all angles are small. Centring removes that shared
offset and reveals how the genes *vary* together. TCF3 (a B-lymphoid transcription factor) and cyclin D3 are both higher
in ALL, CD33 (a myeloid marker) is higher in AML — hence positive and negative correlations.

⚠️ **Pitfall.** "Similarity" computed on uncentred data is dominated by the average level. Always ask whether a method
centres the data — PCA must (`011_PCA` slide 36: "Remember to mean center the data before projection").

✏️ **Your turn.** Standardise both genes (subtract the mean, divide by the standard deviation computed with `np.std`)
and compute `z1 @ z2 / 72`. Which number from the table do you get, and why does dividing by $N$ (not $N-1$) give it
exactly?

---
## 2 · Distances: Euclidean or correlation?

> 📍 **Where we are.** Section 1 gave lengths and angles. A **distance** between two vectors is the length of their
> difference. k-nearest neighbours (Module 08) and clustering (Module 12, `006_clustering` slides 15–18) are built
> entirely on a choice of distance, and the choice changes the biology you find.

$$d_{\text{Euc}}(\mathbf x, \mathbf y) = \|\mathbf x - \mathbf y\| = \sqrt{\sum_{i=1}^{N} (x_i - y_i)^2},
\qquad d_{\text{cor}}(\mathbf x, \mathbf y) = 1 - r(\mathbf x, \mathbf y).$$

*In words:* the **Euclidean distance** is the straight-line distance between the two points and is sensitive to the
overall level and scale of each vector; the **correlation distance** only cares about the *shape* of the profiles
(0 = identical shape, 1 = unrelated, 2 = mirror images).

In [ ]:
# ── 2.1 Same shape, different scale: a SIMULATED cell-cycle time course ──
rng = np.random.default_rng(SEED + 2)
t = np.arange(0, 120, 10)                                # 12 time points (minutes)
wave = np.sin(2 * np.pi * t / 120)                       # one cell-cycle period
genes_tc = {"A: periodic, low level": 5 + 1.0 * wave + rng.normal(0, 0.1, t.size),
            "B: same programme, 3× higher": 8 + 3.0 * wave + rng.normal(0, 0.1, t.size),
            "C: opposite phase, low level": 5 - 1.0 * wave + rng.normal(0, 0.1, t.size)}
A, B, C = genes_tc.values()
print(f"Euclidean distance  A–B = {np.linalg.norm(A - B):5.2f}   A–C = {np.linalg.norm(A - C):5.2f}")
print(f"correlation dist.   A–B = {1 - np.corrcoef(A, B)[0, 1]:5.2f}   A–C = {1 - np.corrcoef(A, C)[0, 1]:5.2f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
for (name, v), c in zip(genes_tc.items(), ["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(t, v, "o-", color=c, label=name, ms=4)
ax.set_xlabel("time after release from arrest (min)"); ax.set_ylabel("expression (a.u.)")
ax.set_title("SIMULATED time course: Euclidean pairs A with C, correlation pairs A with B")
ax.legend(fontsize=7, loc="upper right"); plt.tight_layout(); plt.show()

🔍 **What just happened?** By **Euclidean** distance, gene A is much closer to gene C (about 5.0) than to gene B (about
11.5) — because A and C share the same average level, even though C goes *down* whenever A goes *up*. By **correlation**
distance, A and B are identical in shape (distance ≈ 0) and A and C are mirror images (≈ 2). If your question is "which
genes belong to the same regulatory programme?", correlation distance is usually what you want; if absolute levels
matter (e.g. comparing patients measured on the same scale), Euclidean distance is natural. `006_clustering` slides
15–18 introduce exactly these two measures (Euclidean distance and the correlation coefficient) as the basis of
clustering; this simulated time course shows why the choice between them matters.

### 2.2 Which patients resemble each other?

Now the rows: each patient is a vector of gene expressions. We keep the 500 genes with the largest variance across
patients (chosen **without** looking at the ALL/AML labels) and compute the correlation between every pair of patients.

In [ ]:
# ── 2.2 Patient-by-patient correlation matrix of the Golub data ──
labels = golub["y"]                                       # 0 = ALL, 1 = AML
top = np.argsort(X_g.var(axis=0))[::-1][:500]             # 500 most variable genes (no labels used)
order = np.argsort(labels, kind="stable")                 # ALL patients first, then AML
R = np.corrcoef(X_g[np.ix_(order, top)])                  # 72 × 72: correlation between patient ROWS
lab_sorted = labels[order]
same_all = R[np.ix_(lab_sorted == 0, lab_sorted == 0)][np.triu_indices(47, 1)].mean()
same_aml = R[np.ix_(lab_sorted == 1, lab_sorted == 1)][np.triu_indices(25, 1)].mean()
between = R[np.ix_(lab_sorted == 0, lab_sorted == 1)].mean()
print(f"mean correlation: ALL–ALL {same_all:.3f}   AML–AML {same_aml:.3f}   ALL–AML {between:.3f}")

# nearest neighbour of each patient by correlation distance — does it share the diagnosis?
R_full = np.corrcoef(X_g[:, top]); np.fill_diagonal(R_full, -np.inf)
nn = R_full.argmax(axis=1)
print(f"patients whose most-correlated other patient has the same diagnosis: {np.mean(labels[nn] == labels):.1%}")

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1)
ax.axhline(46.5, color="k", lw=1); ax.axvline(46.5, color="k", lw=1)
ax.set_xticks([23, 59]); ax.set_xticklabels(["ALL (47)", "AML (25)"]); ax.set_yticks([23, 59]); ax.set_yticklabels(["ALL", "AML"])
ax.set_title("Golub: correlation between patients\n(500 most variable genes)")
plt.colorbar(im, ax=ax, fraction=0.046, label="Pearson r"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The 72 × 72 matrix shows a **block structure**: patients correlate more with patients of
the same leukaemia type than with the other type (mean correlation about 0.46 within ALL and 0.49 within AML, but only
0.27 between the types), and for 71 of the 72 patients (98.6 %) the single most-correlated other patient has the same
diagnosis. We never told
the computer which patient had which diagnosis — the labels were only used to sort the rows for display. This is the
geometric fact behind *class discovery* (clustering, Module 12) and *class prediction* by nearest neighbours (Module 08)
in Golub *et al.* (1999).

✏️ **Your turn.** Replace the correlation by Euclidean distance between patient rows
(`np.linalg.norm(Xs[:, None, :] - Xs[None, :, :], axis=2)` with `Xs = X_g[:, top]`) and recompute the nearest-neighbour
agreement. Is it higher or lower? What changes if you first standardise each gene (column)?

---
## 3 · Matrices: data tables you can multiply

> 📍 **Where we are.** We have vectors and distances. A **matrix** is a whole table, and multiplying matrices is how the
> regression slides write "every person's predicted phenotype" in one line.

🧮 **Math refresher — matrices.** A matrix $X$ with $N$ rows and $J$ columns is written $X \in \mathbb R^{N\times J}$
("an $N$ by $J$ table of real numbers"). The entry in row $i$, column $j$ is $X_{ij}$. The 02-620 regression slides
write it $x^{i}_{j}$: **superscript = sample (individual), subscript = feature (SNP)** (`002_regression` slide 26).
Row $i$ is the profile of sample $i$, written $\mathbf x^{i\top}$; column $j$ is feature $j$ across all samples.

In [ ]:
# ── 3.1 Rows, columns and shapes ──
X_p = prostate[FEATURES].to_numpy(float)                  # N = 97 men × J = 8 clinical predictors
y_p = prostate["lpsa"].to_numpy(float)                    # outcome: log PSA
print("prostate X:", X_p.shape, " y:", y_p.shape)
print("row 0 (man 1's 8 measurements):", X_p[0])
print("column 0 (lcavol of the first 6 men):", X_p[:6, 0])
print("Golub: one patient row has", X_g[0].shape[0], "entries; one gene column has", X_g[:, 0].shape[0], "entries")

### 3.2 The matrix–vector product: everyone's score in one line

$$X\mathbf w = \begin{pmatrix} x^{1}_{1} & \cdots & x^{1}_{J} \\ \vdots & & \vdots \\ x^{N}_{1} & \cdots & x^{N}_{J}\end{pmatrix}
\begin{pmatrix} w_1 \\ \vdots \\ w_J \end{pmatrix}
= \begin{pmatrix} \mathbf x^{1\top}\mathbf w \\ \vdots \\ \mathbf x^{N\top}\mathbf w \end{pmatrix}
= w_1 \begin{pmatrix} x^{1}_{1} \\ \vdots \\ x^{N}_{1}\end{pmatrix} + \dots + w_J \begin{pmatrix} x^{1}_{J} \\ \vdots \\ x^{N}_{J}\end{pmatrix}.$$

*In words:* multiplying an $N\times J$ matrix by a $J$-vector gives an $N$-vector. There are two equally true
readings. **Row view:** entry $i$ is the dot product of sample $i$'s profile with the weights — "each person's weighted
sum of genotypes". **Column view:** the result is a weighted sum of the columns — "add up each SNP's genotype column,
scaled by its effect". In genetics, $X\mathbf w$ with $X$ = genotypes (0/1/2) and $\mathbf w$ = per-allele effect sizes
is a **polygenic score**, and it is exactly the prediction $\mathbf x^\top\mathbf w$ of the multivariate regression model
(`002_regression` slides 28–30).

In [ ]:
# ── 3.2 A polygenic score four ways (SIMULATED genotypes and effect sizes) ──
G = np.array([[0, 1, 2, 0],          # 6 individuals × 4 SNPs, genotypes = copies of the T allele
              [1, 1, 0, 2],
              [2, 0, 1, 1],
              [0, 2, 2, 0],
              [1, 0, 0, 1],
              [2, 2, 1, 2]])
w = np.array([1.5, -0.8, 0.0, 2.0])  # effect of each extra T allele on systolic blood pressure (mmHg)

score_loop = np.zeros(6)
for i in range(6):                                   # the definition: row i · w
    for j in range(4):
        score_loop[i] += G[i, j] * w[j]
score_rows = np.array([G[i] @ w for i in range(6)])  # row view: one dot product per person
score_cols = sum(w[j] * G[:, j] for j in range(4))   # column view: weighted sum of SNP columns
score_mat = G @ w                                    # the matrix–vector product
print("loop      :", score_loop)
print("row view  :", score_rows)
print("column view:", score_cols)
print("G @ w     :", score_mat)

# augmenting with a column of ones absorbs the intercept w0 (002_regression slide 31)
X_aug = np.column_stack([np.ones(6), G])             # 6 × 5, first column all ones
w_aug = np.r_[120.0, w]                              # w0 = 120 mmHg baseline
print("\nX_aug =\n", X_aug)
print("predicted blood pressure X_aug @ w_aug:", X_aug @ w_aug)

🔍 **What just happened?** All four computations agree. Check individual 1 by hand:
$0\times1.5 + 1\times(-0.8) + 2\times0 + 0\times2.0 = -0.8$ mmHg. Then we **augmented** the genotype matrix with a
column of ones and put the intercept $w_0 = 120$ on top of $\mathbf w$; now $X_{\text{aug}}\mathbf w_{\text{aug}} = 120 + G\mathbf w$, so individual 1 is predicted at 119.2 mmHg. This is the trick of `002_regression` slide 31:

$$y = \mathbf x^\top\mathbf w + \epsilon = 1\cdot w_0 + x_1 w_1 + \dots + x_J w_J + \epsilon,\qquad
\mathbf x = (1, x_1, \dots, x_J)^\top,\ \mathbf w = (w_0, w_1, \dots, w_J)^\top .$$

*In words:* if every sample gets an extra feature that is always 1, the intercept becomes just another weight, and
the whole model is a single dot product — one set of parameters to differentiate, as the slide's speaker notes say.

### 3.3 Matrix–matrix products and the transpose

$$(AB)_{ik} = \sum_{j} A_{ij} B_{jk}, \qquad A\in\mathbb R^{N\times J},\ B\in\mathbb R^{J\times K} \Rightarrow AB \in \mathbb R^{N\times K};
\qquad (A^\top)_{ij} = A_{ji}, \quad (AB)^\top = B^\top A^\top .$$

*In words:* entry $(i,k)$ of a product is the dot product of row $i$ of $A$ with column $k$ of $B$, so the **inner
sizes must match** ($J$ and $J$) and the outer sizes give the result. The transpose flips a matrix over its diagonal
(rows become columns); transposing a product reverses the order — the rule the slides use on `002_regression`
slide 35 ("when we move the transpose, remember to flip the order").

In [ ]:
# ── 3.3 Shape rules, non-commutativity and the transpose rule ──
A = np.array([[1, 2, 0], [0, 1, 3]])          # 2 × 3
B = np.array([[1, 0], [2, 1], [0, 4]])        # 3 × 2
print("A @ B (2×3 times 3×2 → 2×2):\n", A @ B)
print("B @ A (3×2 times 2×3 → 3×3):\n", B @ A)
print("(A @ B).T equals B.T @ A.T ?", np.array_equal((A @ B).T, B.T @ A.T))
try:
    A @ A                                     # 2×3 times 2×3: inner sizes 3 and 2 do not match
except ValueError as err:
    print("A @ A fails:", str(err)[:80], "…")

🔍 **What just happened?** $AB$ is 2 × 2 but $BA$ is 3 × 3: for matrices, **order matters** ($AB \ne BA$ in general).
The transpose rule holds, and multiplying incompatible shapes raises an error that names the mismatched dimension —
the most common bug in matrix code. When you meet a formula such as $(X^\top X)^{-1}X^\top\mathbf y$, check the shapes
first: $(J\times N)(N\times J) = J\times J$, then $(J\times J)(J\times N)(N\times 1) = J\times 1$ — one weight per feature.

### 3.4 $X^\top X$: the matrix every regression formula contains

$$\big(X^\top X\big)_{jk} = \sum_{i=1}^{N} x^{i}_{j}\, x^{i}_{k}, \qquad \big(X^\top \mathbf y\big)_{j} = \sum_{i=1}^{N} x^{i}_{j}\, y^{i}.$$

*In words:* $X^\top X$ is a $J\times J$ table whose diagonal holds the **sums of squares** of each feature and whose
off-diagonal entries hold the **sums of cross-products** of pairs of features; $X^\top\mathbf y$ holds the
cross-products of each feature with the outcome. If every column has been **centred** (mean subtracted), dividing by
$N$ gives the familiar statistics of Module 02:

$$S = \frac{1}{N} X_c^\top X_c \quad (\text{covariance matrix}), \qquad \frac{1}{N} X_c^\top \mathbf y_c \quad (\text{covariance of each feature with } y).$$

*In words:* the covariance matrix is $X^\top X$ of the centred data divided by $N$ — the definition PCA uses
(`011_PCA`: $S = \frac1N X^\top X$ for mean-centred $X$).

In [ ]:
# ── 3.4 XᵀX, the covariance matrix and Xᵀy on the prostate data ──
N = X_p.shape[0]
Xc = X_p - X_p.mean(axis=0)                  # centre every column (broadcasting, Module 01)
yc = y_p - y_p.mean()
S = Xc.T @ Xc / N                            # 8 × 8 covariance matrix
print("XᵀX shape:", (X_p.T @ X_p).shape, "| S equals np.cov(bias=True)?",
      np.allclose(S, np.cov(X_p, rowvar=False, bias=True)))
sd = np.sqrt(np.diag(S))                     # diagonal = variances
corr = S / np.outer(sd, sd)                  # covariance → correlation
print("\ncorrelation matrix of the predictors (from S):")
print(pd.DataFrame(corr, index=FEATURES, columns=FEATURES).round(2).to_string())
cov_with_y = Xc.T @ yc / N
print("\ncorrelation of each predictor with lpsa:", dict(zip(FEATURES, (cov_with_y / (sd * yc.std())).round(2).tolist())))

🔍 **What just happened?** $X_c^\top X_c/N$ reproduces NumPy's covariance matrix exactly, and dividing by the standard
deviations turns it into the correlation matrix. The last line uses $X_c^\top\mathbf y_c$: log cancer volume (`lcavol`)
has the largest correlation with log PSA (0.73), followed by `svi` and `lcp` — the same message as the eight scatter plots
of `002_regression` slide 42. The predictor correlations matter too: `lcavol`, `svi`, `lcp`, `gleason` and `pgg45` are
correlated with one another (correlations between 0.3 and 0.75), which is why a multivariate model's coefficients can differ from these one-at-a-time
correlations (Module 06).

✏️ **Your turn.** Standardise the columns first (`Z, _, _ = standardize(X_p)`) and compute `Z.T @ Z / N`. Which matrix
do you get? Why does that make the diagonal all ones?

### 📑 Slide connection — `002_regression`, slides 26–31

- **What the slides show.** The phenotype vector $\mathbf y$ (N individuals) and the genotype matrix $X$ (N individuals × J loci, entries
    $x^i_j$) are put into one multivariate model $y = w_0 + \mathbf x^\top\mathbf w + \epsilon$ that estimates each SNP's
    effect *in the presence of all the others* (slides 26–30, with the picture "genotype row × weight column = trait 2.1").
    Slide 31 augments $\mathbf x$ with a leading 1 and $\mathbf w$ with $w_0$, so $y = \mathbf x^\top\mathbf w + \epsilon$.
- **What this notebook adds.** Both readings of $X\mathbf w$ (row = each person's score, column = weighted sum of SNP columns), computed four
    ways and checked by hand; the shape bookkeeping that lets you verify any slide formula; $X^\top X$ and $X^\top\mathbf y$
    as sums of squares and cross-products, and their link to the covariance matrix used later in PCA.
- **Exam tip.** Be able to write the augmented $\mathbf x$, $\mathbf w$ and the $N\times(J+1)$ matrix $X$ (first column all
    ones) and to state the shape of every term in $\mathbf w = (X^\top X)^{-1}X^\top\mathbf y$; a frequent mistake is to
    forget that $X^\top X$ is $(J+1)\times(J+1)$ once the intercept column is added.

---
## 4 · Solving linear systems: identity, inverse and `solve`

> 📍 **Where we are.** Regression ends with the **normal equations** $X^\top X\,\mathbf w = X^\top\mathbf y$
> (`002_regression` slide 38) — a system of linear equations. Here we learn what an inverse is, how to solve such
> systems safely, and meet a biological system you can solve exactly.

🧮 **Math refresher — identity, inverse, determinant.** The **identity matrix** $I$ has ones on the diagonal and zeros
elsewhere; $I\mathbf x = \mathbf x$ for every $\mathbf x$ (it is the matrix version of the number 1). A square matrix $A$
has an **inverse** $A^{-1}$ if $A^{-1}A = AA^{-1} = I$ (the matrix version of $1/a$). For a 2 × 2 matrix

$$A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}, \qquad \det A = ad - bc, \qquad
A^{-1} = \frac{1}{ad-bc}\begin{pmatrix} d & -b \\ -c & a \end{pmatrix}.$$

*In words:* swap the diagonal, negate the off-diagonal, divide by the **determinant** $ad-bc$. If the determinant is
zero, there is no inverse — just as $1/0$ does not exist. Such a matrix is called **singular**.

A **linear system** $A\mathbf f = \mathbf b$ asks: which vector $\mathbf f$, multiplied by the known matrix $A$, gives
the known vector $\mathbf b$? If $A$ is invertible, the unique answer is $\mathbf f = A^{-1}\mathbf b$.

In [ ]:
# ── 4.1 A 2×2 inverse by hand ──
A2 = np.array([[4.0, 2.0], [1.0, 3.0]])
det = A2[0, 0] * A2[1, 1] - A2[0, 1] * A2[1, 0]
A2_inv_hand = np.array([[A2[1, 1], -A2[0, 1]], [-A2[1, 0], A2[0, 0]]]) / det
print("det =", det)
print("inverse by hand:\n", A2_inv_hand)
print("np.linalg.inv agrees?", np.allclose(A2_inv_hand, np.linalg.inv(A2)))
print("A @ A⁻¹ =\n", (A2 @ A2_inv_hand).round(12) + 0.0)        # "+ 0.0" turns a printed -0. into 0.

### 4.2 A biological linear system: how many T cells are in this blood sample?

A **bulk** RNA measurement of blood mixes the transcripts of many cell types. If we know how strongly each marker gene
is expressed in each pure cell type (a **signature matrix** $A$, genes × cell types) and the bulk sample is a mixture with
unknown cell-type fractions $\mathbf f$, then

$$\mathbf b = A\,\mathbf f, \qquad b_g = \sum_{c} A_{gc}\, f_c .$$

*In words:* each marker's bulk level is the fraction-weighted sum of its level in each cell type. With as many
independent markers as cell types, solving the system gives the fractions. Real **deconvolution** tools (for example
CIBERSORT, Newman *et al.* 2015) use hundreds of genes and constrained least squares, but the core is this equation.
The numbers below are SIMULATED.

In [ ]:
# ── 4.2 Cell-type deconvolution with np.linalg.solve (SIMULATED signature matrix) ──
cell_types = ["T cell", "B cell", "monocyte"]
markers = ["CD3E", "CD19", "CD14"]
A_sig = np.array([[100.0,  5.0,  10.0],     # rows = marker genes, columns = pure cell types (a.u.)
                  [  4.0, 80.0,   6.0],
                  [  8.0,  6.0, 120.0]])
f_true = np.array([0.60, 0.15, 0.25])        # true fractions in the bulk sample
b_bulk = A_sig @ f_true                      # what the bulk measurement would show
print("bulk marker levels b =", dict(zip(markers, b_bulk.round(2).tolist())))

f_solve = np.linalg.solve(A_sig, b_bulk)     # solves A f = b without forming A⁻¹
f_inv = np.linalg.inv(A_sig) @ b_bulk        # same answer via the inverse
print("solve:", dict(zip(cell_types, f_solve.round(4).tolist())), "| inv:", f_inv.round(4))

# real measurements are noisy: add 3 % noise and use 6 markers (more equations than unknowns)
rng = np.random.default_rng(SEED + 4)
A6 = np.vstack([A_sig, [[60.0, 3.0, 5.0], [2.0, 45.0, 3.0], [5.0, 4.0, 70.0]]])   # 3 extra markers
b6 = A6 @ f_true * (1 + rng.normal(0, 0.03, 6))
f_ls = np.linalg.solve(A6.T @ A6, A6.T @ b6) # normal equations AᵀA f = Aᵀb (least squares, Module 06)
print("6 noisy markers, least squares:", dict(zip(cell_types, f_ls.round(3).tolist())),
      "| np.linalg.lstsq:", np.linalg.lstsq(A6, b6, rcond=None)[0].round(3))

🔍 **What just happened?** With three markers and three cell types, `np.linalg.solve` recovers the fractions
0.60 / 0.15 / 0.25 exactly, and so does the inverse. With six noisy markers there are **more equations than unknowns**
and no $\mathbf f$ satisfies all of them exactly; instead we solved $A^\top A\,\mathbf f = A^\top\mathbf b$, the
**least-squares** solution, which lands close to the truth and agrees with NumPy's dedicated `lstsq`. That
"multiply both sides by $A^\top$" step is precisely how the regression slides arrive at $X^\top X\mathbf w = X^\top\mathbf y$; Module 04 derives *why* it minimises the squared error.

### 4.3 The normal equations on real data

$$X^\top X\,\mathbf w = X^\top \mathbf y \quad\Longrightarrow\quad \mathbf w = (X^\top X)^{-1} X^\top \mathbf y .$$

*In words:* the regression weights solve a $(J+1)\times(J+1)$ linear system built from the cross-product matrices of
Section 3.4. On the slides it is written with an inverse; in code we hand the system to `solve`.

In [ ]:
# ── 4.3 Solve the normal equations for the prostate training men ──
train = prostate["train"].to_numpy()
Z_tr, mu_tr, sd_tr = standardize(X_p[train])                 # z-score with the TRAINING men's mean/sd
X_tr = np.column_stack([np.ones(train.sum()), Z_tr])         # augmented: 67 × 9
y_tr = y_p[train]

w_solve = np.linalg.solve(X_tr.T @ X_tr, X_tr.T @ y_tr)
w_inv = np.linalg.inv(X_tr.T @ X_tr) @ X_tr.T @ y_tr
sk = LinearRegression().fit(Z_tr, y_tr)
table = pd.DataFrame({"solve": w_solve, "inv": w_inv, "scikit-learn": np.r_[sk.intercept_, sk.coef_]},
                     index=["w0 (intercept)"] + FEATURES)
print(table.round(3).to_string())
resid = y_tr - X_tr @ w_solve
print("\nXᵀ(y − Xw) =", (X_tr.T @ resid).round(10) + 0.0, "→ the residual is orthogonal to every column")

🔍 **What just happened?** Three routes give the same nine weights; `lcavol` has the largest coefficient, as on
`002_regression` slide 43 (Module 06 reproduces the textbook table exactly and interprets every number — here we only
care about the linear algebra). The last line is a geometric fact worth remembering: at the least-squares solution,
$X^\top(\mathbf y - X\mathbf w) = \mathbf 0$ — the residual vector is **perpendicular to every column of $X$**. That is
just the normal equations rearranged, and Section 7 explains why it means "least squares is a projection".

### 4.4 Why `solve` and not `inv`?

Mathematically $A^{-1}\mathbf b$ and `solve(A, b)` are the same. Numerically, `solve` factorises $A$ once and
back-substitutes, which is faster and loses fewer digits to rounding. The cost of either grows like $J^3$ — the
bottleneck the speaker notes of `002_regression` slide 40 warn about ("the cost of cubing this matrix").

In [ ]:
# ── 4.4 Speed and accuracy: solve vs inv ──
def best_time(f, repeats: int = 3) -> float:
    """Fastest of a few runs of f() in seconds (the minimum is the least disturbed by other programs)."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter(); f(); times.append(time.perf_counter() - t0)
    return min(times)


rng = np.random.default_rng(SEED + 44)
for J in [200, 400, 800]:
    M = rng.normal(size=(J + 50, J)); Amat = M.T @ M          # a symmetric invertible J × J matrix (like XᵀX)
    bvec = rng.normal(size=J)
    t_solve = best_time(lambda: np.linalg.solve(Amat, bvec))   # 'lambda: …' = a tiny unnamed function to time
    t_inv = best_time(lambda: np.linalg.inv(Amat) @ bvec)
    print(f"J = {J:4d}: solve {1000 * t_solve:6.1f} ms   inv @ b {1000 * t_inv:6.1f} ms")

n = 12                                                         # a notoriously ill-conditioned matrix (Hilbert)
H = 1.0 / (np.arange(n)[:, None] + np.arange(n)[None, :] + 1)
x_true = np.ones(n); b_h = H @ x_true
for name, x_hat in [("solve", np.linalg.solve(H, b_h)), ("inv", np.linalg.inv(H) @ b_h)]:
    print(f"{name:5s}: residual ‖Hx − b‖ = {np.linalg.norm(H @ x_hat - b_h):.1e}")
print(f"condition number of H: {np.linalg.cond(H):.1e}  (≈ how many times rounding errors can be amplified)")

🔍 **What just happened?** Doubling $J$ multiplies the time by several-fold (ideally ≈ 8 for a $J^3$ algorithm;
timings on a shared machine are noisy, so look at the trend, not the exact milliseconds), and `inv` is typically the
slower route. On the Hilbert matrix — whose **condition number** of about $10^{16}$ means rounding errors can be amplified
up to $10^{16}$-fold — `solve` returns a much smaller residual than multiplying by the computed inverse. Rule of thumb
for the whole course: **write $(X^\top X)^{-1}X^\top\mathbf y$ on paper, call `np.linalg.solve(X.T @ X, X.T @ y)` in
code**. A huge condition number is also a warning sign that the system is *nearly singular* — the topic of Section 5.

✏️ **Your turn.** In Section 4.2, change the B-cell signature column to be almost identical to the T-cell column
(e.g. `[95, 6, 11]`) and re-solve with 3 % noise. How wrong do the fractions become? Compute `np.linalg.cond(A_sig)`
before and after.

### 📑 Slide connection — `002_regression`, slides 32–40

- **What the slides show.** Maximising the Gaussian conditional likelihood is minimising the squared error, written in matrix form as
    $\mathbf w_{MLCE} = \arg\min (\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w)$ with the augmented $N\times(J+1)$
    matrix (slides 32–34). Differentiating gives $X^\top\mathbf y = X^\top X\mathbf w$ and
    $\mathbf w = (X^\top X)^{-1}X^\top\mathbf y$ (slides 35–38); slide 40 notes this is a closed form but that computing and
    inverting the $(J+1)\times(J+1)$ matrix is costly for very large $J$.
- **What this notebook adds.** What "invert" and "solve" mean, a 2 × 2 inverse by hand, a biological linear system (deconvolution) that you can
    solve exactly and by least squares, the normal equations solved three ways on the prostate data, the
    residual-orthogonality check, and a timing/accuracy experiment showing why code uses `solve`. (The derivative step —
    slides 35–39 — is Module 04's job.) The slides abbreviate the estimator both as MLCE and MCLE; both mean "maximum
    conditional likelihood estimate".
- **Exam tip.** Know that $X^\top X$ is symmetric, $(J+1)\times(J+1)$, and that the normal equations say "residual ⟂ every
    column of $X$". If asked about cost: forming $X^\top X$ is $O(NJ^2)$ and solving is $O(J^3)$.

---
## 5 · Rank and singularity: when the equations have no unique answer

> 📍 **Where we are.** Section 4 assumed the matrix was invertible. In genomics it often is not — and
> `002_regression` slide 46 says exactly that: "$X^\top X$ is not invertible if the number of samples $N$ is smaller than
> the number of inputs $J$ (3 million SNPs and 1,000 individuals)". Here we see why.

🧮 **Math refresher — linear independence and rank.** Columns $\mathbf x_1, \dots, \mathbf x_J$ are **linearly
dependent** if one of them can be written as a weighted sum of the others (e.g. $\mathbf x_3 = 2\mathbf x_1 - \mathbf x_2$); otherwise they are **independent**. The **rank** of a matrix is the number of independent columns (equivalently,
rows), so

$$\operatorname{rank}(X) \le \min(N, J), \qquad \operatorname{rank}(X^\top X) = \operatorname{rank}(X).$$

*In words:* an $N\times J$ table can never contain more than $N$ independent columns (each column lives in an
$N$-dimensional space) nor more than $J$; and $X^\top X$ inherits exactly the rank of $X$. A $J\times J$ matrix is
invertible **only if its rank is $J$** ("full rank"). If some column is a combination of others, the regression cannot
tell their effects apart, and infinitely many weight vectors fit equally well.

In [ ]:
# ── 5.1 Perfect linkage disequilibrium and the allele-coding trap (SIMULATED genotypes) ──
rng = np.random.default_rng(SEED + 5)
G4, _, _ = simulate_genotypes(rng, n_samples=50, n_snps=4)
G_ld = np.column_stack([G4, G4[:, 1]]).astype(float)      # SNP 5 is an exact copy of SNP 2 (perfect LD)
X_ld = np.column_stack([np.ones(50), G_ld])               # intercept + 5 SNP columns
XtX = X_ld.T @ X_ld
print("perfect LD: columns =", X_ld.shape[1], "| rank(X) =", np.linalg.matrix_rank(X_ld),
      "| det(XᵀX) =", f"{np.linalg.det(XtX):.2e}")
print("eigenvalues of XᵀX:", np.linalg.eigvalsh(XtX).round(6) + 0.0)
y_ld = G4 @ np.array([0.5, 1.0, 0.0, -0.3]) + rng.normal(0, 1, 50)
try:
    w_bad = np.linalg.solve(XtX, X_ld.T @ y_ld)
    print("solve returned (meaningless) weights:", w_bad.round(1))
except np.linalg.LinAlgError as err:
    print("np.linalg.solve refused:", err)

x_A = 2 - G4[:, 0]                                         # copies of the OTHER allele at SNP 1
X_trap = np.column_stack([np.ones(50), G4[:, 0], x_A])    # intercept, T-count, A-count
print("\nallele-coding trap: rank of [1, x_T, x_A] =", np.linalg.matrix_rank(X_trap), "of 3 columns",
      "because x_T + x_A = 2 × (column of ones)")

🔍 **What just happened?** Copying SNP 2 into a fifth column (two SNPs in **perfect linkage disequilibrium** — always
inherited together) leaves 6 columns but only rank 5. The determinant of $X^\top X$ is zero and so is one of its
eigenvalues (up to rounding), and here `np.linalg.solve` refuses with "Singular matrix". (With other data — or even on
another computer — rounding errors can make such a matrix only *almost* singular; then `solve` returns huge, meaningless
weights instead of an error — a silent failure that is worse than a loud one. The cell prints whichever happens.) The model cannot know whether the effect belongs to SNP 2 or to its copy: any split of the effect between them
fits equally well.

The second example is a classic beginner's trap: coding *both* alleles at a SNP (copies of T and copies of A) together
with an intercept. Because $x_T + x_A = 2$ for every person, the three columns are dependent and $X^\top X$ is singular.
Statistics packages silently drop a column in this situation; NumPy will not.

### 5.2 Fewer people than SNPs: $N < J$

In [ ]:
# ── 5.2 N = 5 individuals, J = 10 SNPs: infinitely many perfect fits (SIMULATED) ──
rng = np.random.default_rng(SEED + 52)
G_small, _, _ = simulate_genotypes(rng, n_samples=5, n_snps=10)
X_s = G_small.astype(float)
y_s = rng.normal(0, 1, 5)                                  # phenotypes of the 5 people (pure noise!)
print("X is", X_s.shape, "| rank(X) =", np.linalg.matrix_rank(X_s), "| XᵀX is", (X_s.T @ X_s).shape,
      "with rank", np.linalg.matrix_rank(X_s.T @ X_s))
print("eigenvalues of XᵀX:", np.linalg.eigvalsh(X_s.T @ X_s).round(8) + 0.0)

w1 = np.linalg.lstsq(X_s, y_s, rcond=None)[0]              # one exact solution (the shortest one)
null_dir = np.linalg.svd(X_s)[2][-1]                       # a direction v with X v = 0 (from the SVD)
w2 = w1 + 3.0 * null_dir                                   # another exact solution
for name, wv in [("w1", w1), ("w2", w2)]:
    print(f"{name}: training error ‖y − Xw‖ = {np.linalg.norm(y_s - X_s @ wv):.1e}   weights {wv.round(2)}")
x_new = np.array([1, 0, 2, 1, 1, 0, 2, 1, 0, 1], float)    # a 6th person
print(f"prediction for a new person: w1 → {x_new @ w1:+.2f},  w2 → {x_new @ w2:+.2f}")

🔍 **What just happened?** With 5 people and 10 SNPs, $X^\top X$ is 10 × 10 but has rank 5: half of its eigenvalues are
zero. Two completely different weight vectors $\mathbf w_1$ and $\mathbf w_2$ both reproduce the five phenotypes
**perfectly** (training error ≈ $10^{-15}$, i.e. zero up to rounding) — even though the phenotypes were pure noise! —
yet they disagree about a new person. A perfect fit to training data that tells you nothing reliable about new data is
the essence of **overfitting**, and it is guaranteed when $N < J$. The direction `null_dir` satisfies $X\mathbf v = \mathbf 0$: adding any multiple of it to the weights changes nothing the data can see. (We borrowed it from the singular
value decomposition, `np.linalg.svd`, which is the subject of a 🚀 Bonus item.)

In [ ]:
# ── 5.3 Rank of XᵀX and its smallest eigenvalue as the number of people grows (J = 10 SNPs) ──
rng = np.random.default_rng(SEED + 53)
J = 10
G30, _, _ = simulate_genotypes(rng, n_samples=30, n_snps=J, maf_range=(0.25, 0.5))   # common SNPs
Ns = np.arange(2, 31)
ranks, smallest = [], []
for N_ in Ns:
    Gn = G30[:N_].astype(float)                    # the first N people of the same cohort
    XtX_n = Gn.T @ Gn
    ranks.append(np.linalg.matrix_rank(XtX_n))
    smallest.append(max(np.linalg.eigvalsh(XtX_n)[0], 1e-16))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
axes[0].plot(Ns, ranks, "o-", ms=3); axes[0].plot(Ns, np.minimum(Ns, J), "k--", lw=1, label="min(N, J)")
axes[0].set_xlabel("number of individuals N"); axes[0].set_ylabel("rank of XᵀX"); axes[0].legend()
axes[0].set_title("rank(XᵀX) = min(N, J) for J = 10 SNPs")
axes[1].semilogy(Ns, smallest, "o-", ms=3, color="tab:red"); axes[1].axvline(J, color="k", ls="--", lw=1)
axes[1].set_xlabel("number of individuals N"); axes[1].set_ylabel("smallest eigenvalue (log)")
axes[1].set_title("XᵀX becomes invertible only once N ≥ J (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** We added people one at a time to the same simulated cohort. The rank of $X^\top X$ follows
$\min(N, J)$, and its smallest eigenvalue jumps from numerical zero (about $10^{-15}$) to a clearly positive value once
there are as many people as SNPs. (With genotypes, which take only the values 0/1/2, it can occasionally take a person
or two more — for instance if a SNP is not yet polymorphic among the first few people — so rerun with another seed and
look at the region just after $N = J$.) With 3 million SNPs and 1,000 people, $X^\top X$ is a 3,000,000 × 3,000,000 matrix of rank at most
1,000: hopelessly singular.

⚠️ **Misconception.** "Singular" is not a software bug you can fix with a better solver. It means the **data do not
contain enough information** to determine all the weights. The remedies change the *question*: add prior information
(MAP / ridge, Section 8 and Module 07), assume few non-zero effects (lasso, Module 07), test one SNP at a time (GWAS,
Module 06), or reduce dimension (PCA, Module 14).

✏️ **Your turn.** In 5.2, add two more people (N = 7). Is $X^\top X$ still singular? How many zero eigenvalues does it
have now? Predict the answer before running.

### 📑 Slide connection — `002_regression`, slides 44–46

- **What the slides show.** Slide 44 asks for the limitations of the MLE; slide 46 answers: good news — a closed-form estimate
    $\mathbf w = (X^\top X)^{-1}X^\top\mathbf y$; bad news — $X^\top X$ is not invertible when $N < J$ ("3 million SNPs and
    1,000 individuals"); remedy — MAP estimation. The speaker notes explain non-invertibility by a zero determinant or a
    rank smaller than $J$ ("a fat matrix").
- **What this notebook adds.** A from-scratch account of rank and linear dependence; three concrete ways to get a singular $X^\top X$
    (perfect LD, coding both alleles with an intercept, $N<J$); a demonstration that $N<J$ produces infinitely many perfect
    fits that disagree on new individuals; and the eigenvalue view (zero eigenvalues ⇔ singular), which Section 8 uses to
    show why the MAP/ridge fix works.
- **Exam tip.** Exam favourite: "Why can't we use the closed-form least-squares solution for GWAS?" — answer with rank:
    $\operatorname{rank}(X^\top X) = \operatorname{rank}(X) \le N < J$, so the $J\times J$ matrix is singular; mention that
    even when $N \ge J$, highly correlated SNPs make it *nearly* singular (huge condition number, unstable weights).

---
## 6 · Eigenvalues and eigenvectors: the natural axes of a matrix

> 📍 **Where we are.** Section 5 used eigenvalues as a singularity detector. Now we meet them properly. PCA
> (`011_PCA` slides 17–31) is nothing but the eigendecomposition of the covariance matrix $S$ of Section 3.4, and
> Gaussian graphical models (Module 18) and spectral arguments in ridge regression rely on the same idea.

🧮 **Math refresher — eigenvectors.** Multiplying a vector by a square matrix usually changes both its length and its
direction. A non-zero vector $\mathbf u$ whose direction does **not** change is an **eigenvector** of $A$; it is only
stretched (or shrunk, or flipped) by a factor $\lambda$, the **eigenvalue**:

$$A\,\mathbf u = \lambda\,\mathbf u .$$

*In words:* "applying $A$ to $\mathbf u$ is the same as multiplying $\mathbf u$ by the number $\lambda$". ("Eigen" is
German for "own": these are the matrix's own directions.)

For a **symmetric** matrix ($A = A^\top$, such as $X^\top X$ or any covariance matrix) three facts make life easy:
the eigenvalues are real numbers; eigenvectors of different eigenvalues are **orthogonal**; and the matrix can be
rebuilt from them (`011_PCA` slides 18–20):

$$S = U D U^\top, \qquad U = [\mathbf u_1, \dots, \mathbf u_K],\quad D = \operatorname{diag}(\lambda_1, \dots, \lambda_K),\quad U^\top U = I .$$

*In words:* stack the unit-length eigenvectors as the columns of $U$ and put the eigenvalues on the diagonal of $D$;
then $S$ = rotate into the eigen-axes ($U^\top$), stretch each axis by its eigenvalue ($D$), rotate back ($U$). $U^\top U = I$ says the columns of $U$ are unit vectors that are mutually perpendicular — an **orthogonal matrix**. (The PCA deck
calls the number of features $K$ instead of $J$.)

In [ ]:
# ── 6.1 Which directions does a symmetric matrix leave unchanged? ──
S2 = np.array([[2.0, 1.2], [1.2, 1.0]])
evals, U2 = np.linalg.eigh(S2)                       # eigh: for symmetric matrices, eigenvalues in ASCENDING order
U2 = U2 * np.sign(U2[0])                             # eigenvectors are defined up to sign; make first entries positive
print("eigenvalues (ascending):", evals.round(4))
print("eigenvectors (columns of U):\n", U2.round(4))
for k in range(2):
    print(f"S u{k + 1} = {(S2 @ U2[:, k]).round(4)}   λ{k + 1} u{k + 1} = {(evals[k] * U2[:, k]).round(4)}")

angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
circle = np.c_[np.cos(angles), np.sin(angles)]       # 24 unit vectors
images = circle @ S2.T                               # S applied to each one (row-wise S u)
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for c, im in zip(circle, images):
    ax.plot([c[0], im[0]], [c[1], im[1]], color="0.75", lw=0.8)
ax.plot(*circle.T, "o", ms=3, color="tab:blue", label="unit vectors u")
ax.plot(*images.T, "o", ms=3, color="tab:orange", label="S u")
for k, col in zip(range(2), ["tab:green", "tab:red"]):
    ax.annotate("", xy=evals[k] * U2[:, k], xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=col, lw=2.5))
    ax.text(*(evals[k] * U2[:, k] * 1.1 + [0.08, -0.25]), f"λ{k + 1}u{k + 1}", color=col, fontsize=10)
ax.set_xlim(-2.8, 2.8); ax.set_ylim(-2.2, 2.2)
ax.set_aspect("equal"); ax.legend(fontsize=7, loc="lower right")
ax.set_title("S turns the circle into an ellipse;\nonly eigenvectors keep their direction")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each grey segment joins a unit vector (blue) to its image $S\mathbf u$ (orange). Most
segments change direction, but along the two eigenvector directions the image lies on the same line: $S\mathbf u_k = \lambda_k\mathbf u_k$, verified numerically in the printout. The circle becomes an **ellipse** whose long axis is the
eigenvector with the larger eigenvalue ($\lambda_2 = 2.8$ here) and whose short axis is the other one
($\lambda_1 = 0.2$). (We flipped the eigenvectors' signs so that their first entries are positive; see the pitfall below.)

⚠️ **Pitfall (`011_PCA` slides 18–20 say "Call `numpy.linalg.eigh`").** `np.linalg.eigh` returns eigenvalues in
**ascending** order, so the *largest* eigenvalue — the first principal component — is the **last** column. Always sort
in descending order before you talk about "PC1". Also, an eigenvector is only defined up to its sign: $-\mathbf u$ is
just as valid, so different programs may print opposite signs.

### 6.2 Eigenvalues of a 2 × 2 matrix by hand

$$\det(S - \lambda I) = 0 \;\Longleftrightarrow\; (a-\lambda)(d-\lambda) - b^2 = 0 \;\Longleftrightarrow\;
\lambda^2 - (a + d)\lambda + (ad - b^2) = 0, \qquad S = \begin{pmatrix} a & b\\ b & d\end{pmatrix}.$$

*In words:* $S\mathbf u = \lambda\mathbf u$ means $(S-\lambda I)\mathbf u = \mathbf 0$ for a non-zero $\mathbf u$,
which is only possible if $S-\lambda I$ is singular (Section 5), i.e. its determinant is zero. For a 2 × 2 matrix
that is a quadratic equation in $\lambda$. Its two roots add up to $a+d$ (the **trace**) and multiply to $ad-b^2$ (the
determinant).

In [ ]:
# ── 6.2 Solve the characteristic equation with the quadratic formula ──
a_, b_, d_ = S2[0, 0], S2[0, 1], S2[1, 1]
tr, dt = a_ + d_, a_ * d_ - b_ ** 2
lam_hand = np.array([(tr - np.sqrt(tr ** 2 - 4 * dt)) / 2, (tr + np.sqrt(tr ** 2 - 4 * dt)) / 2])
print("by hand:", lam_hand.round(4), "| eigh:", evals.round(4))
u_big = np.array([b_, lam_hand[1] - a_]); u_big /= np.linalg.norm(u_big)     # (S − λI)u = 0 → u ∝ (b, λ − a)
print("eigenvector of the larger eigenvalue by hand:", u_big.round(4), "| eigh:", U2[:, 1].round(4))
print(f"sum of eigenvalues = {evals.sum():.4f} = trace {tr:.4f};  product = {evals.prod():.4f} = det {dt:.4f}")

🔍 **What just happened?** The quadratic formula gives $\lambda = (3 \mp \sqrt{9 - 4\times0.56})/2 = 0.2$ and $2.8$,
exactly what `eigh` returned, and the same eigenvector (eigenvectors are only defined up to sign, so another program
might print $-\mathbf u$; both are correct). The sum of the eigenvalues equals the trace and their product equals the determinant: a zero
eigenvalue therefore means a zero determinant — the singularity of Section 5 seen from the eigen side.

### 6.3 The covariance ellipse of two co-expressed genes

Now a real covariance matrix: TCF3 and cyclin D3 across all 72 patients. If we project the centred data onto a unit
vector $\mathbf u$, the projected values $X_c\mathbf u$ have variance

$$\operatorname{Var}(X_c\mathbf u) = \frac{1}{N}(X_c\mathbf u)^\top(X_c\mathbf u) = \mathbf u^\top S\,\mathbf u,
\qquad \text{and for an eigenvector: } \mathbf u_k^\top S\,\mathbf u_k = \lambda_k .$$

*In words:* the variance of the data along any direction is $\mathbf u^\top S\mathbf u$, and along an eigenvector it is
exactly the eigenvalue. That is why `011_PCA` slide 27 labels the eigenvalues "variance explained".

In [ ]:
# ── 6.3 Eigendecomposition of a real 2 × 2 covariance matrix (TCF3 and CCND3, 72 patients) ──
X2 = X_g[:, [GENE_INDEX["M31523_at"], GENE_INDEX["M92287_at"]]]
X2c = X2 - X2.mean(axis=0)
S_g = X2c.T @ X2c / X2c.shape[0]
lam, U_g = np.linalg.eigh(S_g)
lam, U_g = lam[::-1], U_g[:, ::-1]                        # sort DESCENDING: column 0 = largest = PC1
print("covariance S =\n", S_g.round(4))
print("eigenvalues (descending):", lam.round(4))
print("U D Uᵀ reproduces S?", np.allclose(U_g @ np.diag(lam) @ U_g.T, S_g), "| UᵀU = I?", np.allclose(U_g.T @ U_g, np.eye(2)))
for k in range(2):
    print(f"variance of the data projected on u{k + 1}: {np.var(X2c @ U_g[:, k]):.4f}  (λ{k + 1} = {lam[k]:.4f})")
print(f"total variance: trace(S) = {np.trace(S_g):.4f} = λ1 + λ2 = {lam.sum():.4f};  PC1 share = {lam[0] / lam.sum():.1%}")

theta = np.linspace(0, 2 * np.pi, 200)
ellipse = (U_g @ np.diag(2 * np.sqrt(lam)) @ np.vstack([np.cos(theta), np.sin(theta)])).T + X2.mean(axis=0)
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for c, name in [(0, "ALL"), (1, "AML")]:
    ax.scatter(*X2[labels == c].T, s=16, color=["tab:blue", "tab:red"][c], label=name, alpha=0.8)
ax.plot(*ellipse.T, "k-", lw=1, label="2-σ ellipse from S")
for k, col in zip(range(2), ["tab:green", "tab:purple"]):
    tip = X2.mean(axis=0) + 2 * np.sqrt(lam[k]) * U_g[:, k]
    ax.annotate("", xy=tip, xytext=X2.mean(axis=0), arrowprops=dict(arrowstyle="-|>", color=col, lw=2.5))
    ax.text(*tip, f" u{k + 1}", color=col, fontsize=10)
ax.set_xlabel("TCF3 expression (log10)"); ax.set_ylabel("CCND3 expression (log10)"); ax.set_aspect("equal")
ax.set_title("eigenvectors of S = axes of the data cloud"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** The two eigenvectors of the 2 × 2 covariance matrix are the axes of the ellipse that
summarises the cloud of patients; their lengths are drawn as $2\sqrt{\lambda_k}$ (two standard deviations). Projecting
the data on $\mathbf u_1$ gives a variance equal to $\lambda_1$, and on $\mathbf u_2$ a variance equal to $\lambda_2$,
exactly as the formula promised; the eigenvalues add up to the total variance (the trace), and the first axis carries
most of it (see the printed share). Along $\mathbf u_1$ — "both genes high" vs "both genes low" — the ALL and AML patients
separate. This 2-D picture is the toy of `011_PCA` slides 24–28, built from real leukaemia data.

### 6.4 The spectrum of a big covariance matrix

In [ ]:
# ── 6.4 Eigenvalues of the 500 × 500 covariance matrix of the 500 most variable Golub genes ──
Xtop = X_g[:, top]
Xtop_c = Xtop - Xtop.mean(axis=0)
S_big = Xtop_c.T @ Xtop_c / Xtop_c.shape[0]               # 500 × 500
lam_big = np.linalg.eigvalsh(S_big)[::-1]                 # descending
n_pos = int(np.sum(lam_big > 1e-10 * lam_big[0]))
print("S is", S_big.shape, "| number of clearly non-zero eigenvalues:", n_pos, "(N − 1 =", Xtop.shape[0] - 1, ")")
print(f"share of total variance in the top 2 eigenvalues: {lam_big[:2].sum() / lam_big.sum():.1%}; top 10: {lam_big[:10].sum() / lam_big.sum():.1%}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(np.arange(1, 81), np.maximum(lam_big[:80], 1e-18), "o-", ms=3)
ax.axvline(71.5, color="tab:red", ls="--", lw=1); ax.text(52, lam_big[0] / 3, "N − 1 = 71", color="tab:red", fontsize=8)
ax.set_xlabel("eigenvalue number (largest first)"); ax.set_ylabel("eigenvalue (log scale)")
ax.set_title("Golub, 500 genes: a few large eigenvalues, then a cliff after 71"); plt.tight_layout(); plt.show()

🔍 **What just happened?** A 500 × 500 covariance matrix estimated from 72 patients has only **71** non-zero
eigenvalues: centring uses up one degree of freedom, so the rank is at most $N - 1$ (Section 5 again — this matrix is
singular too!). Among the non-zero ones, a handful are much larger than the rest: the first two capture a sizeable share
of all variance (about 30 %) and the first ten close to two-thirds (about 64 %). That steep decline is what makes PCA useful — a few
directions summarise thousands of genes (Module 14 will look at which patients and genes drive them).

✏️ **Your turn.** Repeat 6.3 with TCF3 and CD33 (`"M23197_at"`), which are negatively correlated. In which quadrant
does $\mathbf u_1$ point now? What happens to the eigenvectors if you multiply one gene by 10 (a change of units)?

### 📑 Slide connection — `011_PCA_mz_031825`, slides 18–20 (and 24–28)

- **What the slides show.** A review of eigendecomposition: a $K\times K$ symmetric matrix $S = U D U^\top$ with $U = [\mathbf u_1, \dots, \mathbf u_K]$ orthogonal ($U^\top U = I$, eigenvectors as columns) and $D$ diagonal, computed with
    `numpy.linalg.eigh`; slides 24–28 apply it to a 2 × 2 sample covariance, label $\mathbf u_1, \mathbf u_2$ as PC 1 and
    PC 2 and the eigenvalues $\lambda_1, \lambda_2$ as the "variance explained".
- **What this notebook adds.** What an eigenvector *is* (the unchanged-direction picture), the 2 × 2 characteristic equation by hand, the
    trace/determinant checks, the proof-by-computation that $\mathbf u^\top S\mathbf u$ is the variance along
    $\mathbf u$ and equals $\lambda_k$ on an eigenvector, a real two-gene covariance ellipse, and the rank-$(N-1)$ cliff of
    a big covariance matrix. Note the deck's $K$ = our $J$ (number of features).
- **Exam tip.** Remember that `eigh` sorts eigenvalues **ascending** (reverse before picking PCs), that eigenvector signs are
    arbitrary, and be ready to verify $U^\top U = I$ and $S = UDU^\top$ in code. A covariance matrix's eigenvalues are
    never negative (they are variances).

---
## 7 · Projections and new coordinates

> 📍 **Where we are.** Eigenvectors give a new set of perpendicular axes. Describing data in those axes — and
> throwing some away — is step 2 of PCA (`011_PCA` slides 32–40). The same idea of projection explains least squares.

🧮 **Math refresher — orthonormal bases and coordinates.** Two unit vectors $\mathbf u_1, \mathbf u_2$ that are
perpendicular form an **orthonormal basis** of the plane: every point can be reached by walking some number of steps
along $\mathbf u_1$ and some along $\mathbf u_2$. Because the axes are perpendicular unit vectors, the numbers of steps
are just dot products:

$$\mathbf x = c_1\mathbf u_1 + c_2\mathbf u_2, \qquad c_k = \mathbf x^\top\mathbf u_k .$$

*In words:* the coordinate of $\mathbf x$ along an axis is the length of its shadow on that axis. In the usual axes
$\mathbf u_1 = (1,0)$, $\mathbf u_2=(0,1)$, the point $\mathbf x = (2,1)$ means "walk 2 steps along $\mathbf u_1$ and 1
step along $\mathbf u_2$" (`011_PCA` slide 37). Keeping only $c_1$ replaces $\mathbf x$ by its **projection**
$\mathbf x_p = c_1\mathbf u_1$ (slide 38).

In [ ]:
# ── 7.1 The point x = (2, 1) in the standard axes and in rotated axes (011_PCA slides 37–38) ──
x_pt = np.array([2.0, 1.0])
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
r1, r2 = np.array([1.0, 1.0]) / np.sqrt(2), np.array([-1.0, 1.0]) / np.sqrt(2)   # rotated by 45°
print("standard axes: coordinates", (float(x_pt @ e1), float(x_pt @ e2)), "| e1ᵀe2 =", e1 @ e2)
c1, c2 = x_pt @ r1, x_pt @ r2
print(f"rotated axes : coordinates ({c1:.3f}, {c2:.3f})  | r1ᵀr2 = {r1 @ r2:.1f}")
print("reconstruction c1 r1 + c2 r2 =", (c1 * r1 + c2 * r2).round(10))
x_proj = c1 * r1
print("projection onto r1 alone: x_p =", x_proj.round(3), "| (x − x_p)ᵀ r1 =", round(float((x_pt - x_proj) @ r1), 12))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (u_a, u_b), title, off in [(axes[0], (e1, e2), "usual axes: x = 2·u1 + 1·u2", (-0.6, -0.35)),
                                    (axes[1], (r1, r2), f"rotated axes: x = {c1:.2f}·u1 {c2:+.2f}·u2", (-1.9, 0.25))]:
    ax.axhline(0, color="0.85", lw=0.8); ax.axvline(0, color="0.85", lw=0.8)
    for u_, col, nm in [(u_a, "tab:blue", "u1"), (u_b, "tab:green", "u2")]:
        ax.plot([-2.5 * u_[0], 2.8 * u_[0]], [-2.5 * u_[1], 2.8 * u_[1]], color=col, lw=0.6, alpha=0.6)
        ax.annotate("", xy=u_, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=col, lw=2)); ax.text(*(u_ * 1.1), nm, color=col)
    ax.plot(*x_pt, "ko"); ax.text(x_pt[0] + 0.08, x_pt[1] - 0.2, "x = (2, 1)")
    cA = x_pt @ u_a
    ax.plot(*(cA * u_a), "s", color="tab:red"); ax.plot([x_pt[0], cA * u_a[0]], [x_pt[1], cA * u_a[1]], "r--", lw=1)
    ax.text(cA * u_a[0] + off[0], cA * u_a[1] + off[1], f"x_p: {cA:.2f} steps along u1", color="tab:red", fontsize=8)
    ax.set_xlim(-1, 3); ax.set_ylim(-1, 2.5); ax.set_aspect("equal"); ax.set_title(title)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the usual axes, $\mathbf x = (2,1)$ is 2 steps along $\mathbf u_1$ and 1 along
$\mathbf u_2$; its projection on $\mathbf u_1$ is $(2,0)$, as on slide 37. In axes rotated by 45°, the same point is
$3/\sqrt2 \approx 2.12$ steps along the new $\mathbf u_1$ and $-1/\sqrt2 \approx -0.71$ along the new $\mathbf u_2$;
walking those steps brings us back exactly to $(2,1)$. Its projection on the new $\mathbf u_1$ is $(1.5, 1.5)$, and the
leftover $\mathbf x - \mathbf x_p$ is perpendicular to $\mathbf u_1$. Slide 38 says it in words: "walk in the direction
of $\mathbf u_1$ by $X^\top \mathbf u_1$ steps".

### 7.2 All samples at once: PCA in miniature

$$Z = X_c\,U, \qquad X_c = Z\,U^\top, \qquad X_p = X_c\,[\mathbf u_1, \dots, \mathbf u_M] \;\;(N\times M).$$

*In words:* multiplying the centred data matrix by $U$ computes every sample's coordinates along every eigen-axis in
one product (row $i$ of $Z$ holds $\mathbf x^{i\top}\mathbf u_1, \dots, \mathbf x^{i\top}\mathbf u_K$); multiplying by
$U^\top$ walks back. Keeping only the first $M$ columns of $U$ gives the reduced $N\times M$ data of `011_PCA` slide 40.

In [ ]:
# ── 7.2 New coordinates for the two-gene Golub data ──
Z2 = X2c @ U_g                                            # 72 × 2 coordinates along u1, u2
print("covariance of the new coordinates:\n", (Z2.T @ Z2 / len(Z2)).round(5) + 0.0, "\n(diagonal = eigenvalues", lam.round(5), ")")
print("walk back: Z Uᵀ == X_c ?", np.allclose(Z2 @ U_g.T, X2c))
X_1d = np.outer(Z2[:, 0], U_g[:, 0])                      # keep only coordinate 1 → back in gene space
mse = np.mean(np.sum((X2c - X_1d) ** 2, axis=1))
print(f"keeping 1 of 2 coordinates: mean squared reconstruction error = {mse:.4f} = λ2 = {lam[1]:.4f}")

🔍 **What just happened?** In the eigen-coordinates the two new variables are **uncorrelated** (the off-diagonal of
their covariance is zero) and their variances are the eigenvalues — the "decorrelation" PCA promises on `011_PCA`
slide 54. Walking back with $U^\top$ recovers the data exactly, and keeping only the first coordinate loses an average
squared error exactly equal to the discarded eigenvalue $\lambda_2$. That is the whole logic of choosing how many PCs to
keep: the eigenvalues you drop are the variance you lose.

### 7.3 Least squares is a projection

In Section 4.3 the residual was perpendicular to every column of $X$. Geometrically, the fitted values $\hat{\mathbf y} = X\mathbf w$ are the **projection of $\mathbf y$ onto the space spanned by the columns of $X$**:

$$\hat{\mathbf y} = X(X^\top X)^{-1}X^\top\mathbf y = P\,\mathbf y, \qquad P^2 = P, \qquad X^\top(\mathbf y - P\mathbf y) = \mathbf 0 .$$

*In words:* among all vectors that can be written as $X\mathbf w$, least squares picks the one closest to $\mathbf y$,
which is its shadow; the "projection matrix" $P$ (statisticians call it the hat matrix) does nothing more when applied
twice, because a shadow's shadow is itself.

In [ ]:
# ── 7.3 The projection ("hat") matrix of the prostate training regression ──
P = X_tr @ np.linalg.solve(X_tr.T @ X_tr, X_tr.T)        # 67 × 67
y_hat = P @ y_tr
print("P is", P.shape, "| P @ P == P ?", np.allclose(P @ P, P), "| trace(P) =", round(np.trace(P), 6), "= number of weights")
print("ŷ from P equals X w:", np.allclose(y_hat, X_tr @ w_solve))
print("‖y‖² =", round(y_tr @ y_tr, 3), " = ‖ŷ‖² + ‖y − ŷ‖² =", round(y_hat @ y_hat + (y_tr - y_hat) @ (y_tr - y_hat), 3), "(Pythagoras)")

🔍 **What just happened?** $P$ is idempotent ($P^2 = P$), it reproduces the fitted values, and its trace equals the
number of fitted weights (9: intercept + 8 predictors) — a quantity statisticians call the model's "degrees of
freedom", which reappears when models of different sizes are compared (Module 23). Pythagoras holds in 67 dimensions because the fit and the residual are perpendicular. You now have three
equivalent descriptions of least squares: minimise squared error (Module 04), solve the normal equations (Section 4),
project onto the column space (here).

✏️ **Your turn.** In 7.1, project $\mathbf x = (2,1)$ onto the direction $(1,2)/\sqrt5$. Compute the coordinate by hand
first ($\mathbf x^\top\mathbf u = 4/\sqrt5$), then check in code.

### 📑 Slide connection — `011_PCA_mz_031825`, slides 32–40 (especially 37–38)

- **What the slides show.** Step 2 of PCA projects each (mean-centred) sample onto the principal components by inner products,
    $\mathbf x^\top\mathbf u_1 = \|\mathbf x\|\cos\theta$ for a unit vector (slides 32–36). Slides 37–38 explain coordinates:
    $\mathbf x = (2,1)$ means walking 2 steps along $\mathbf u_1 = (1,0)$ and 1 along $\mathbf u_2 = (0,1)$, and in the
    eigenvector coordinate system you walk $X^\top\mathbf u_1$ steps along $\mathbf u_1$; slide 40 assembles the reduced
    data $X_p = X[\mathbf u_1 \dots \mathbf u_M]$ ($N\times M$).
- **What this notebook adds.** The rotated-basis version of the $(2,1)$ example with exact numbers ($3/\sqrt2$, $-1/\sqrt2$) and a picture,
    reconstruction from coordinates, the proof-by-computation that eigen-coordinates are uncorrelated with variances
    $\lambda_k$ and that dropping a coordinate costs exactly its eigenvalue, and the bonus insight that least squares is
    also a projection.
- **Exam tip.** Know why centring is required before projecting (otherwise PC1 tends to point at the mean, not along the
    spread) and be able to compute a projection $\mathbf x^\top\mathbf u$ and the projected point $(\mathbf x^\top \mathbf u)\mathbf u$ by hand; remember $\mathbf u$ must have length 1.

---
## 8 · Adding $\lambda I$: a one-line cure for singularity

> 📍 **Where we are.** Section 5 showed $X^\top X$ is singular when $N<J$; Section 6 showed singular ⇔ a zero
> eigenvalue. Here is the trick that the MAP estimate of `002_regression` slides 47–54 uses, seen from linear algebra.

If $X^\top X\,\mathbf u_k = \lambda_k\mathbf u_k$, then for any number $\lambda > 0$

$$(X^\top X + \lambda I)\,\mathbf u_k = X^\top X\mathbf u_k + \lambda\mathbf u_k = (\lambda_k + \lambda)\,\mathbf u_k .$$

*In words:* adding $\lambda$ to every diagonal entry keeps the eigenvectors and raises **every eigenvalue by
$\lambda$**. Eigenvalues of $X^\top X$ are never negative, so after the shift all of them are at least $\lambda > 0$: no
zero eigenvalue, hence invertible — "Invertible, even if $N<J$" (`002_regression` slide 54). The resulting estimate
$\mathbf w = (X^\top X + \lambda I)^{-1}X^\top\mathbf y$ is **ridge regression** (Module 07), which the slides derive as
the MAP estimate with a Gaussian prior on $\mathbf w$ (Module 05 explains MAP).

In [ ]:
# ── 8.1 Shift the eigenvalues of the singular 10 × 10 XᵀX from Section 5.2 ──
XtX_s = X_s.T @ X_s
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
axes[0].semilogy(np.arange(1, 11), np.maximum(np.linalg.eigvalsh(XtX_s), 1e-17), "o-", label="XᵀX")
for lam_r, col in [(0.1, "tab:orange"), (1.0, "tab:green")]:
    ev = np.linalg.eigvalsh(XtX_s + lam_r * np.eye(10))
    axes[0].semilogy(np.arange(1, 11), ev, "o-", color=col, label=f"XᵀX + {lam_r} I")
    print(f"λ = {lam_r}: smallest eigenvalue {ev[0]:.3f}, condition number {np.linalg.cond(XtX_s + lam_r * np.eye(10)):.1f}")
axes[0].set_xlabel("eigenvalue number (ascending)"); axes[0].set_ylabel("eigenvalue (log)"); axes[0].legend(fontsize=7)
axes[0].set_title("adding λI lifts every eigenvalue by λ")

lams = np.logspace(-3, 2, 30)
W = np.array([np.linalg.solve(XtX_s + l * np.eye(10), X_s.T @ y_s) for l in lams])
axes[1].semilogx(lams, W, lw=1); axes[1].axhline(0, color="k", lw=0.6)
axes[1].set_xlabel("λ (log scale)"); axes[1].set_ylabel("weight"); axes[1].set_title("unique weights for every λ > 0;\nlarger λ pulls them toward 0")
plt.tight_layout(); plt.show()
w_r = np.linalg.solve(XtX_s + 0.1 * np.eye(10), X_s.T @ y_s)
print("λ = 0.1 → unique weights:", w_r.round(2), "| training error ‖y − Xw‖ =", round(float(np.linalg.norm(y_s - X_s @ w_r)), 3))

🔍 **What just happened?** The five zero eigenvalues of the singular $X^\top X$ (blue, at the numerical floor) jump to
exactly $\lambda$ after adding $\lambda I$, and the non-zero ones move up by the same amount; the condition number becomes
finite and modest. Now `solve` returns a **unique** answer for every $\lambda>0$, and as $\lambda$ grows the weights are
pulled toward zero (right panel) — the "shrinkage" of ridge regression. The fit no longer passes exactly through the five
phenotypes (training error > 0): we trade a perfect but arbitrary fit for a stable one. How to choose $\lambda$, and why
this is the same as putting a prior on $\mathbf w$, are the topics of Modules 05, 07 and 23.

✏️ **Your turn.** Verify the eigenvector claim: compute `np.linalg.eigh` of `XtX_s` and of `XtX_s + 0.5 * np.eye(10)`
and check that the eigenvector matrices agree (up to sign) and that the eigenvalues differ by exactly 0.5.

## 📌 Key takeaways

- A biological data set is a matrix $X$ with **N rows = samples** and **J columns = features**; a gene's profile across
  patients and a patient's profile across genes are both **vectors**.
- The **dot product** $\mathbf x^\top\mathbf y$ measures alignment; divided by the lengths it is the cosine of an angle.
  **Pearson correlation is the cosine of the angle between centred vectors** — centring matters.
- **Euclidean** distance compares levels, **correlation** distance compares shapes; the choice changes which genes or
  patients look alike (k-NN, clustering).
- $X\mathbf w$ is every sample's weighted sum of features (a polygenic score, the regression prediction); a column of
  ones absorbs the intercept. $X^\top X$ holds sums of squares and cross-products; for centred data $S = X^\top X/N$ is
  the covariance matrix.
- Least squares solves the **normal equations** $X^\top X\mathbf w = X^\top\mathbf y$; use `np.linalg.solve`, not an
  explicit inverse. The residual is orthogonal to every column — least squares is a **projection**.
- $X^\top X$ is **singular** when columns are dependent (perfect LD, coding both alleles with an intercept) and always
  when **$N<J$**: $\operatorname{rank}(X^\top X) \le \min(N,J)$, and infinitely many weight vectors fit perfectly.
- A symmetric matrix factorises as $S = UDU^\top$ with orthonormal eigenvectors; eigenvalues are the **variances along
  the eigen-axes** (`eigh` sorts them ascending!). Projecting onto the top eigenvectors is PCA.
- Adding $\lambda I$ raises every eigenvalue by $\lambda$, making $X^\top X + \lambda I$ invertible even when $N<J$ —
  the linear-algebra heart of ridge regression.

---
## ✅ Checkpoint

1. Compute $\|(3, 4)\|$ and the dot product of $(1, 2, 3)$ and $(4, -5, 6)$ by hand.
   <details><summary>Answer</summary>$\sqrt{9+16} = 5$; $1\cdot4 + 2\cdot(-5) + 3\cdot6 = 4 - 10 + 18 = 12$.</details>
2. Two genes have a raw-vector cosine of 0.99 but a Pearson correlation of −0.3. How is that possible?
   <details><summary>Answer</summary>Log-expression values are all positive, so the raw vectors point into the same region of space and their angle is small whatever their co-variation. Pearson's $r$ is the cosine of the **centred** vectors, which removes the common offset and measures whether the genes go up and down together — here they tend to move in opposite directions.</details>
3. $X$ holds 100 patients × 20 genes, $\mathbf w$ has 20 entries and $\mathbf y$ has 100. Give the shapes of $X\mathbf w$, $X^\top X$, $XX^\top$ and $X^\top\mathbf y$.
   <details><summary>Answer</summary>$X\mathbf w$: 100 (one score per patient); $X^\top X$: 20 × 20; $XX^\top$: 100 × 100 (patient-by-patient cross-products); $X^\top\mathbf y$: 20 (one cross-product per gene).</details>
4. Why does a design matrix with columns [1, copies of allele T, copies of allele A] make $X^\top X$ singular?
   <details><summary>Answer</summary>For a diploid genotype $x_T + x_A = 2$ for everybody, so the T column plus the A column equals 2 × the intercept column. The three columns are linearly dependent, $\operatorname{rank}(X) = 2 < 3$, hence $X^\top X$ (3 × 3) has rank 2 and no inverse.</details>
5. A GWAS has $N = 1{,}000$ people and $J = 3{,}000{,}000$ SNPs. What are the size and the maximum rank of $X^\top X$, and what does that imply?
   <details><summary>Answer</summary>$X^\top X$ is $3\times10^6 \times 3\times10^6$ but $\operatorname{rank}(X^\top X) = \operatorname{rank}(X) \le 1{,}000$. It is singular, so $(X^\top X)^{-1}$ does not exist and infinitely many weight vectors fit the training data equally well (`002_regression` slide 46) — we need priors/regularisation, sparsity, or one-SNP-at-a-time tests.</details>
6. Give the eigenvalues and eigenvectors of $S = \begin{pmatrix}4 & 0\\ 0 & 1\end{pmatrix}$. Which direction has the largest variance?
   <details><summary>Answer</summary>$\lambda = 4$ with $\mathbf u = (1,0)$ and $\lambda = 1$ with $\mathbf u = (0,1)$ (a diagonal matrix's eigenvectors are the coordinate axes). The first coordinate has variance 4, the largest, so it is PC1.</details>
7. `np.linalg.eigh(S)` returns `w = [0.3, 2.7]` and a matrix `U`. Which column of `U` is the first principal component, and what fraction of the variance does it explain?
   <details><summary>Answer</summary>The **last** column, `U[:, 1]`, because `eigh` sorts eigenvalues in ascending order. It explains $2.7/(0.3+2.7) = 90\%$.</details>
8. What are the coordinates of $\mathbf x = (2, 1)$ in the orthonormal basis $\mathbf u_1 = (1,1)/\sqrt2$, $\mathbf u_2 = (-1, 1)/\sqrt2$, and what is its projection onto $\mathbf u_1$?
   <details><summary>Answer</summary>$c_1 = \mathbf x^\top\mathbf u_1 = 3/\sqrt2 \approx 2.12$, $c_2 = \mathbf x^\top\mathbf u_2 = -1/\sqrt2 \approx -0.71$; the projection is $c_1\mathbf u_1 = (1.5, 1.5)$.</details>

## 🧠 Questions & discussion

1. You want to cluster genes into co-regulated programmes, and separately to cluster patients. Would you use the same distance for both? What preprocessing (centring, standardising genes) would you apply in each case, and why?
2. Perfect LD is rare, but SNPs with $r^2 = 0.98$ are common. What happens to the eigenvalues and the condition number of $X^\top X$, and to the stability of the regression weights, when two SNP columns are almost identical?
3. Why can the eigenvalues of a covariance matrix never be negative? (Hint: $\mathbf u^\top S\mathbf u$ is a variance.) Why can they be zero?
4. If every gene is standardised before computing $X^\top X/N$, you get the correlation matrix. When would PCA on the correlation matrix be preferable to PCA on the covariance matrix for expression data?
5. In the deconvolution example, what goes wrong if two cell types have almost the same signature? Relate your answer to rank, condition number and to the biology of closely related cell states.

## 🚀 Bonus — going further

- **Singular value decomposition.** Every matrix factorises as $X = U\Sigma V^\top$ (`np.linalg.svd`). Show numerically that the eigenvalues of $X^\top X$ are the squared singular values and its eigenvectors are the columns of $V$. Module 14 explains why PCA is usually computed this way when $J \gg N$.
- **Condition numbers.** Generate two SNP columns with correlation $\rho$ (0, 0.5, 0.9, 0.99, 0.999), compute `np.linalg.cond(X.T @ X)` and the spread of the least-squares weights over 200 simulated phenotypes. Plot both against $\rho$.
- **QR instead of normal equations.** `np.linalg.lstsq` avoids forming $X^\top X$ (which squares the condition number). Compare its accuracy with `solve(X.T @ X, X.T @ y)` on a nearly collinear design.
- **Regularised regression in genomics.** The repository's [Regularized linear regression](../../RegularizedLinearRegression/) tutorial shows the $p \gg n$ singularity and the ridge/lasso remedies on real wheat genotypes.
- **Visual intuition.** Grant Sanderson's *Essence of linear algebra* video series (https://www.3blue1brown.com/topics/linear-algebra) animates vectors, matrix products, determinants and eigenvectors — an excellent companion to Sections 1–6.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            03 · Vectors and matrices
Input:             Golub expression matrix (72 × 3,571, log10); prostate table (97 × 8 + lpsa); SIMULATED genotypes
Output:            correlations/distances, covariance matrices, normal-equation weights, ranks, eigen-decompositions
Method:            dot products and norms; X @ w; XᵀX and S = XcᵀXc/N; np.linalg.solve; matrix_rank; np.linalg.eigh;
                   projections Z = Xc U; adding λI
Key equation:      S = U D Uᵀ with UᵀU = I;  Var(Xc u) = uᵀ S u = λ for an eigenvector u
Key parameters:    number of genes kept (500), N vs J in the rank demo, λ in Section 8
What changed when I modified …  (e.g. Euclidean vs correlation distance; N = 7 instead of 5; λ = 0.01 vs 10)
One failure mode:  XᵀX singular (perfect LD, both alleles + intercept, N < J) → no unique least-squares solution
Connection to 02-620 slides:  002_regression 26–40 (data matrix, normal equations), 46 (N < J);
                              011_PCA 18–20 (eigendecomposition), 37–38 (coordinates and projection)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, decks `002_regression` and `011_PCA_mz_031825` (course design S. Kim; instructors M. Xu and M. Zhang).
- Strang G. (2016) *Introduction to Linear Algebra*, 5th ed. Wellesley-Cambridge Press — chapters 1–2 (vectors, matrices, elimination), 3 (rank), 4 (orthogonality, projections, least squares), 6 (eigenvalues).
- Petersen K. B., Pedersen M. S. (2012) *The Matrix Cookbook* (version of 15 November 2012) — the reference linked on `002_regression` slide 35.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §7.3 (least squares and its geometric interpretation), §12.2 (PCA).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §3.2 (linear regression, the hat matrix), §14.5.1 (principal components).
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87 (the preprocessing used here).
- Stamey T. A. *et al.* (1989) Prostate specific antigen in the diagnosis and treatment of adenocarcinoma of the prostate. II. Radical prostatectomy treated patients. *Journal of Urology* 141:1076–1083.
- Newman A. M. *et al.* (2015) Robust enumeration of cell subsets from tissue expression profiles. *Nature Methods* 12:453–457 (CIBERSORT, a real deconvolution method).
- Novembre J. *et al.* (2008) Genes mirror geography within Europe. *Nature* 456:98–101.

## ➡️ Next

**[Module 04 · ⛰️ Calculus and optimization](04_calculus_and_optimization.ipynb)** — you can now *write* the regression
problem in matrix form; next you learn how to *solve* it: derivatives, gradients, gradient descent, convexity and
Lagrange multipliers, including the derivative step of `002_regression` slides 35–39 that turns
$(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w)$ into the normal equations, and the constrained problem whose
answer is "the top eigenvector" (PCA).